# ForenSynth-X+ | Entity Resolution Pipeline
**Version:** 4.0.0

LLM-powered neuro-symbolic entity resolution for multi-modal forensic observations.

## Pipeline (12 stages)
Intake → Normalization → Blocking → Feature Computation → Scoring → Classification → Graph → Clustering → Attachment → Conflict Detection → Entity Labeling → Packaging

## v4 highlights
- **EntityCoreferenceAgent** (Groq LLM): links different aliases across modalities
- **ContextScoringAgent** (Groq LLM): semantic content similarity with batching
- Heuristic fallback when `GROQ_API_KEY` is absent
- Cross-modal clustering for timeline-ready canonical entities


> **Patch note (Timeline Agent contract fix):** this notebook now emits a
> `conflicts` list (full records: `type`, `cluster_id`, `detail`, ...) in both
> the pipeline's output dict and in `timeline_agent_payload`, in addition to
> the existing `conflicts_detected` count. Previously only the count was
> exposed, which made it structurally impossible for the Timeline Agent to
> know which observations a conflict affected - conflict-aware confidence
> penalties silently never fired downstream. `conflicts_detected` is kept
> unchanged for backward compatibility.

## 1. Install dependencies


In [ ]:
%pip install -q rapidfuzz groq networkx sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.4 MB/s eta 0:00:00


## 2. Imports & API key setup


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import logging
import os
import re
import time
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, FrozenSet, List, Optional, Set, Tuple

import networkx as nx
from rapidfuzz import fuzz

try:
    from groq import Groq as _GroqClient  # type: ignore
    _GROQ_AVAILABLE = True
except ImportError:
    _GROQ_AVAILABLE = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s – %(message)s",
)
log = logging.getLogger("forensynth_x_plus")

# Portable API key: Colab secret OR local env var
if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata  # type: ignore
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    except Exception:
        pass


## 2b. Shared normalization module (timestamps, aliases, modality/role, content, action tags, location)

In [ ]:
# ===========================================================================
# SHARED NORMALIZATION MODULE (identical copy also used by the Timeline Agent)
# ===========================================================================
# This cell is byte-for-byte the same normalization.py shipped with the
# Timeline Agent package. It is the SINGLE source of truth for timestamp
# parsing, alias/modality/role normalization, content cleanup, action-tag
# extraction, location normalization, AND (new) the batch normalize_case()
# / NormalizedObservationStore repository abstraction - the seam both
# agents now go through instead of normalizing piecemeal.
#
# If you change this cell, copy the SAME change into
# timeline_agent/normalization.py (or vice versa) to keep both in sync.

"""
ForenSynth-X+ - shared normalization utilities.

Used by BOTH the Entity Resolution pipeline and the Timeline Agent, so the
two agents can never silently disagree about how a timestamp, alias,
modality, role, or piece of content gets interpreted. This file is the
single source of truth for all of that.

Design constraints:
  - Pure stdlib, zero dependencies - so it can be pasted as a single cell
    into the ER Colab notebook AND imported normally by the Timeline Agent
    package without adding an install step either place.
  - Every function is pure (no I/O, no globals mutated) so it's trivially
    unit-testable and safe to run twice on the same data (idempotent).

Grounding note: `ACTION_TAG_FRAGMENTS` below was built by reading the
ACTUAL phrase banks in the generator's templates.py / observations.py
(both ATM_Robbery and Office_Theft domains) rather than guessed - see
tests/test_normalization.py, which checks real generator phrases resolve
to the expected tag. It will not catch every possible paraphrase (the
generator's phrase banks are large and stylistically varied by design -
that's the noise-injection point), so this remains a best-effort heuristic
layer. Anything it can't tag is reported as an unresolved pair by the
Timeline Agent's temporal/causal reasoners rather than silently guessed.
"""
from __future__ import annotations

import re
from abc import ABC, abstractmethod
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Set, Tuple

# ==============================================================================
# Timestamps
# ==============================================================================
# FIX: previously ER (_parse_ts, 4 formats) and the Timeline Agent
# (parse_epoch, 10 formats) each maintained their own format list. Anything
# ER couldn't parse silently became epoch=0 in canonical_entities'
# earliest/latest_timestamp even if the Timeline Agent's more permissive
# parser WOULD have understood it - a real cross-agent inconsistency risk.
# This is now the one parser both agents call.

_TS_FORMATS_TZ = [
    "%Y-%m-%dT%H:%M:%S.%f%z",
    "%Y-%m-%dT%H:%M:%S%z",
]
_TS_FORMATS_NAIVE = [
    "%Y-%m-%dT%H:%M:%S.%f",
    "%Y-%m-%dT%H:%M:%S",
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%d %H:%M",
    "%Y-%m-%d",
    "%d/%m/%Y %H:%M:%S",
    "%d/%m/%Y %H:%M",
    "%m/%d/%Y %H:%M:%S",
    "%m/%d/%Y %H:%M",
]


def parse_timestamp(timestamp: Optional[str]) -> float:
    """Parse a timestamp string to a POSIX epoch float. Returns 0.0 on failure."""
    if not timestamp:
        return 0.0
    ts = timestamp.strip()
    if not ts:
        return 0.0

    if not ts.endswith("Z"):
        for fmt in _TS_FORMATS_TZ:
            try:
                return datetime.strptime(ts, fmt).timestamp()
            except ValueError:
                continue

    ts_clean = re.sub(r"Z$", "", ts).strip()
    for fmt in _TS_FORMATS_NAIVE:
        try:
            dt = datetime.strptime(ts_clean, fmt)
            return dt.replace(tzinfo=timezone.utc).timestamp()
        except ValueError:
            continue

    # Last-resort fallback: Python's own ISO parser catches a few additional
    # well-formed variants on 3.11+; harmless no-op on older versions.
    try:
        dt = datetime.fromisoformat(ts_clean)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.timestamp()
    except Exception:
        pass

    return 0.0


def epoch_to_iso(epoch: float) -> str:
    if epoch <= 0:
        return ""
    return datetime.fromtimestamp(epoch, tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


# ==============================================================================
# Alias normalization
# ==============================================================================

def normalize_alias(alias: Optional[str]) -> str:
    """Lowercase and replace non-alphanumeric chars with underscore."""
    if not alias:
        return "unknown"
    return re.sub(r"[^a-z0-9_]", "_", alias.strip().lower()) or "unknown"


# ==============================================================================
# Modality canonicalization
# ==============================================================================
# FIX: previously modality strings were compared as-is (case-folded only).
# A Field Agent emitting "CCTV" or "phone_call" instead of "video"/"audio"
# would silently fall into the 0.50-reliability "unknown" bucket instead of
# being recognised. This gives modality a real, extensible synonym table.

_MODALITY_SYNONYMS = {
    "video": "video", "cctv": "video", "camera": "video", "footage": "video", "visual": "video",
    "audio": "audio", "voice": "audio", "call": "audio", "phone": "audio", "recording": "audio",
    "text": "text", "sms": "text", "email": "text", "chat": "text", "log": "text", "message": "text",
    "network": "network", "ip": "network", "firewall": "network", "system_log": "network",
}


def normalize_modality(modality: Optional[str]) -> str:
    key = (modality or "").strip().lower()
    return _MODALITY_SYNONYMS.get(key, key or "unknown")


# ==============================================================================
# Role canonicalization
# ==============================================================================

_ROLE_SYNONYMS = {
    "suspect": "suspect", "perpetrator": "suspect", "accused": "suspect", "offender": "suspect",
    "witness": "witness", "bystander": "witness", "eyewitness": "witness",
    "observer": "witness", "reporter": "witness",
    "victim": "victim",
    "system": "system", "sensor": "system", "device": "system",
}


def normalize_role(role: Optional[str]) -> str:
    key = (role or "").strip().lower()
    return _ROLE_SYNONYMS.get(key, key or "unknown")


# ==============================================================================
# Content cleanup
# ==============================================================================

_BOILERPLATE_PATTERNS = [
    re.compile(r"\(captured on footage\)", re.IGNORECASE),
    re.compile(r"\(captured on camera\)", re.IGNORECASE),
    re.compile(r"\(on footage\)", re.IGNORECASE),
    re.compile(r"\(cctv footage\)", re.IGNORECASE),
]


def clean_content(content: Optional[str]) -> str:
    """Strip known template boilerplate and collapse whitespace. Preserves case."""
    text = content or ""
    for pat in _BOILERPLATE_PATTERNS:
        text = pat.sub("", text)
    return re.sub(r"\s+", " ", text).strip()


# ==============================================================================
# Action tag extraction
# ==============================================================================
# FIX: the original CAUSAL_ACTION_RULES matched single exact-root words
# ("enter", "exit", "withdraw"...) against tokenized content. Checked against
# your real CASE_ATM_001 observations, 6 of 7 matched ZERO rule keywords,
# because the generator's phrase banks use inflections ("leaves") and free
# paraphrasing ("fiddling with", "ran out") rather than root-form verbs -
# and this is by design (the generator injects "semantic paraphrasing" as a
# noise type). ACTION_TAG_FRAGMENTS below was built by reading the actual
# generator phrase banks (templates.py description_templates for video,
# observations.py suspect_phrases for audio, both ATM_Robbery and
# Office_Theft domains) so it recognises the phrasing your data actually
# contains, not an idealized vocabulary.
#
# This is fragment/substring matching, not single-token matching, because
# the signal is often multi-word ("steps into", "walks out of", "ran out")
# rather than a single verb.

ACTION_TAG_FRAGMENTS: dict[str, tuple[str, ...]] = {
    "APPROACH": (
        "approach", "walks towards", "walking toward", "walking towards", "nearing",
        "heading in the direction", "heading to", "heading there", "moving towards",
        "moving toward", "almost there", "almost at", "on my way to", "close to the",
        "at the corner", "about to move",
    ),
    "ENTER": (
        "enter", "steps into", "step inside", "stepped in", "pushes open", "pulls open",
        "proceeds inside", "crossing the threshold", "in the booth", "inside the atm",
        "inside the building", "inside the office", "got in", "i'm in", "access card worked",
        "badge scanned", "badge worked", "access granted", "in. door shut", "booth clear",
        "lobby empty", "lobby now",
    ),
    "WITHDRAW": (
        "withdraw", "transaction", "operate the atm", "operating the atm", "pressing keys",
        "inserts card", "insertion area", "conducting activity", "conducting transaction",
        "dispensing", "cash is coming out", "cash out", "getting the money",
        "machine is dispensing", "transaction complete", "transaction going through",
        "card working", "card's in", "processing",
    ),
    "TAMPER": (
        "tamper", "skimmer", "device is attached", "device placed", "reader fitted",
        "reader's set", "rigged", "looks factory", "looks stock", "fitted",
    ),
    "LOITER": (
        "loiter", "lingers", "linger", "remains in", "standing near", "waiting for the right moment",
        "still waiting", "no apparent reason", "hanging around", "keeping an eye out",
    ),
    "EXIT": (
        "exit", "leaves", "leaving", "left the", "left through", "walks out", "walked out",
        "walk out", "departs", "departing", "departed", "recorded leaving", "coming out",
        "come out", "out of the atm", "out of the booth", "out now", "clean exit",
        "exiting via", "i'm out", "i'm through",
    ),
    "FLEE": (
        "flee", "fled", "fleeing", "run", "ran out", "rushed out", "move move move",
        "get out of here", "split up", "abort", "get out fast", "walk away briskly",
        "walked away quickly",
    ),
    "STEAL": (
        "steal", "stole", "stolen", "got the files", "got the folders", "documents are in the bag",
        "grabbed everything", "items secured", "usb is in", "usb full", "copying", "copy complete",
        "data's transferring", "transfer complete", "transfer at", "files are on the drive",
        "wiping the logs", "wiping history",
    ),
    "NAVIGATE": (
        "navigate", "corridor", "taking the stairs", "taking stairs", "server room",
        "restricted section", "target floor", "passing the main hall",
    ),
    "WORK": (
        "perform_legit_work", "finishing up the report", "in the meeting", "sending the last",
        "wrapping up", "at my desk", "on a call with the client", "filing the",
        "catching up on", "working late",
    ),
    "COMMUNICATE": (
        "call me", "reach out", "reaching out", "confirm when ready", "check in", "you ready",
        "we move as planned", "signal is", "meeting point", "text", "message", "sms", "email",
    ),
    "CONFIRM": (
        "confirm", "confirmed", "good to go", "all set", "we're aligned", "understood",
        "copy that", "all clear", "proceeding as agreed",
    ),
    "OBSERVE": (
        "observe", "observed", "witness", "witnessed", "saw", "noticed", "sees someone",
        "bystander", "eyewitness", "i saw", "i noticed",
    ),
    "REPORT": (
        "report", "file a complaint", "speak to an officer", "calling to report",
        "want to let you know", "should report", "should know",
    ),
    "INTERCEPT": (
        "intercept", "intercepted", "misdirected", "not meant for", "picked up a communication",
    ),
}


_NEGATION_WORDS = (
    "not", "n't", "never", "didn't", "doesn't", "wasn't", "isn't", "no ",
    "denies", "denied", "without",
)
_NEGATION_WINDOW_CHARS = 25  # look this far back from the fragment match


def extract_action_tags(content: Optional[str]) -> Set[str]:
    """
    Return the set of canonical action tags detected in `content`, using
    substring/fragment matching grounded in the generator's real phrase
    banks. Best-effort: absence of a tag does not mean "no action happened",
    only "this heuristic layer didn't recognise the phrasing" - callers
    should treat a fully-empty result as a genuinely ambiguous case, not a
    negative signal.

    Negation-aware: a fragment preceded closely by a negation word ("did
    not go inside", "never entered") is NOT tagged. This matters
    specifically for a forensic tool - a suspect's denial ("I did not go
    inside that booth") must never silently become an inferred ENTER
    action; that would fabricate evidence, not just miss it.
    """
    text = clean_content(content).lower()
    if not text:
        return set()
    tags: Set[str] = set()
    for tag, fragments in ACTION_TAG_FRAGMENTS.items():
        for frag in fragments:
            idx = text.find(frag)
            if idx == -1:
                continue
            window_start = max(0, idx - _NEGATION_WINDOW_CHARS)
            preceding = text[window_start:idx]
            if any(neg in preceding for neg in _NEGATION_WORDS):
                continue  # negated - do not tag
            tags.add(tag)
            break
    return tags


# ==============================================================================
# Location normalization
# ==============================================================================
# FIX: location strings are free-text and verbose by generator design
# (e.g. "ATM booth interior, card reader and keypad area"). Comparing them
# with plain lower/strip means two observations at "the same place" in
# different phrasing never register as a location match. `location_key`
# gives a coarser, more matchable token (text before the first comma) while
# `normalized` keeps the full string for display/audit.

def normalize_location(location: Optional[str]) -> Tuple[str, str]:
    """Returns (normalized_full_string, coarse_location_key)."""
    text = (location or "").strip()
    normalized = re.sub(r"\s+", " ", text).lower()
    key = normalized.split(",")[0].strip()
    return normalized, key


# ==============================================================================
# Batch normalization + repository abstraction (Memory Store swap point)
# ==============================================================================
# FIX: previously ER and the Timeline Agent each called the primitive
# functions above (normalize_alias, normalize_role, parse_timestamp, ...)
# piecemeal, scattered across their own intake code - same underlying
# logic (good, that was the earlier fix), but re-derived independently by
# each agent on every run, with no single place representing "the
# normalized form of this case's observations."
#
# NormalizedObservation + normalize_case() below is that single place: ONE
# batch function that takes a case's raw observations and returns the
# fully-normalized form, computed once per call. NormalizedObservationStore
# wraps it behind the same repository-pattern interface Timeline Agent's
# repositories.py already uses elsewhere, so today's "just compute it
# in-process, no persistence" implementation
# (LocalNormalizedObservationStore) and tomorrow's real Memory-Store-backed
# implementation are interchangeable - callers only ever talk to the
# NormalizedObservationStore interface, never to normalize_case() directly,
# so swapping the backing implementation later touches ONE line per agent
# (the place the store gets constructed), not the intake logic itself.
#
# Honest scope note on "storage": LocalNormalizedObservationStore is
# explicitly NOT persistent - it's an in-process dict cache that dies with
# the Python process (a notebook kernel restart, a fresh script run). It
# exists so that if the SAME case gets normalized twice within one run
# (uncommon today, but possible), the second call is free. It is a
# stand-in for the real Memory Store's normalized_observations table, not
# a replacement for it - persistence across runs/processes is exactly the
# gap the real Memory Store is meant to close.


@dataclass
class NormalizedObservation:
    """
    The canonical, fully-normalized form of one observation - the single
    intermediate representation both agents should build their own
    domain objects (ER's Observation, Timeline Agent's RawObservation)
    from, instead of each re-deriving normalization independently.
    """
    obs_id: str
    entity_raw: str
    entity_norm: str
    role: str                    # canonicalized (normalize_role)
    modality: str                 # canonicalized (normalize_modality)
    location_raw: str
    location_key: str
    content_raw: str
    content_clean: str            # boilerplate-stripped, whitespace-collapsed
    timestamp_raw: str
    ts_epoch: float
    time_offset_sec: int          # relative to this case's earliest valid timestamp
    confidence: float
    action_tags: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "NormalizedObservation":
        return cls(**d)


def normalize_observation(raw: Dict[str, Any], base_epoch: float = 0.0) -> NormalizedObservation:
    """Normalize a single raw observation dict. `base_epoch` should be the
    case's earliest valid epoch (see normalize_case) - pass 0.0 only if
    computing time_offset_sec doesn't matter for the caller."""
    entity_raw = str(raw.get("entity", ""))
    timestamp_raw = str(raw.get("timestamp", ""))
    ts_epoch = parse_timestamp(timestamp_raw)
    location_raw = str(raw.get("location", ""))
    _, location_key = normalize_location(location_raw)
    content_raw = str(raw.get("content", ""))
    content_clean = clean_content(content_raw)

    try:
        confidence = max(0.0, min(1.0, float(raw.get("confidence", 0.5))))
    except (TypeError, ValueError):
        confidence = 0.5

    return NormalizedObservation(
        obs_id=str(raw.get("obs_id", "")),
        entity_raw=entity_raw,
        entity_norm=normalize_alias(entity_raw),
        role=normalize_role(raw.get("role", "unknown")),
        modality=normalize_modality(raw.get("modality", "unknown")),
        location_raw=location_raw,
        location_key=location_key,
        content_raw=content_raw,
        content_clean=content_clean,
        timestamp_raw=timestamp_raw,
        ts_epoch=ts_epoch,
        time_offset_sec=int(ts_epoch - base_epoch) if ts_epoch > 0 and base_epoch > 0 else 0,
        confidence=confidence,
        action_tags=sorted(extract_action_tags(content_raw)),
    )


def normalize_case(raw_observations: List[Dict[str, Any]]) -> List[NormalizedObservation]:
    """
    Batch entry point: normalize every observation in a case in one call.
    Two passes (matches the logic ER's _stage_2_normalization already used,
    now centralized): first parse every timestamp, compute the case's
    earliest valid epoch, then normalize each observation with that shared
    base so time_offset_sec is consistent across the whole case.

    Pure function - no dedup, no filtering, one row in maps to one row out.
    Dedup semantics differ meaningfully between ER (content-hash dedup) and
    Timeline Agent (duplicate-obs_id dedup), so that stays each caller's
    own concern, applied to this function's output.
    """
    epochs = [parse_timestamp(str(o.get("timestamp", ""))) for o in raw_observations]
    valid = [e for e in epochs if e > 0]
    base_epoch = min(valid) if valid else 0.0
    return [normalize_observation(o, base_epoch=base_epoch) for o in raw_observations]


class NormalizedObservationStore(ABC):
    """
    Repository-pattern interface for fetching a case's normalized
    observations. Both agents should depend on THIS interface, never call
    normalize_case() directly - that's what makes the Memory Store swap a
    one-class change later.
    """

    @abstractmethod
    def get(self, case_id: str, raw_observations: List[Dict[str, Any]]) -> List[NormalizedObservation]:
        ...


class LocalNormalizedObservationStore(NormalizedObservationStore):
    """
    TODAY's implementation - no real persistence, no Memory Store. Computes
    normalize_case() on first request for a given (case_id, observation
    content) pair within this process, then serves an in-memory cache for
    the rest of the process's lifetime. Cache key is content-aware (a hash
    of the raw observations, not just case_id) - a real bug this caught
    during testing: two different observation sets sharing the same
    case_id (plausible in practice too, e.g. re-processing a case after
    correcting evidence) must never silently return the OTHER set's stale
    cached result. This is intentionally NOT durable across processes (a
    new kernel/process starts with an empty cache) - it exists to avoid
    redundant recomputation within one run, not to replace the real store.

    SWAP POINT: once the team's Memory Store is ready, implement a
    MemoryStoreNormalizedObservationStore(NormalizedObservationStore) that
    reads pre-computed rows from the store's normalized_observations table
    (or writes them via normalize_case() on first access, if the store
    itself owns computing them) instead of caching in-process. Every
    caller here talks only to the `.get()` interface, so no intake code
    in either agent needs to change - only which store gets constructed.
    """

    def __init__(self) -> None:
        self._cache: Dict[Tuple[str, str], List[NormalizedObservation]] = {}

    @staticmethod
    def _content_key(raw_observations: List[Dict[str, Any]]) -> str:
        import hashlib
        import json
        blob = json.dumps(raw_observations, sort_keys=True, default=str)
        return hashlib.sha256(blob.encode("utf-8")).hexdigest()

    def get(self, case_id: str, raw_observations: List[Dict[str, Any]]) -> List[NormalizedObservation]:
        cache_key = (case_id, self._content_key(raw_observations))
        if cache_key in self._cache:
            return self._cache[cache_key]
        result = normalize_case(raw_observations)
        self._cache[cache_key] = result
        return result


_default_store: Optional[NormalizedObservationStore] = None


def get_normalized_observation_store() -> NormalizedObservationStore:
    """
    Module-level default store, matching the same singleton-getter pattern
    used for the semantic similarity scorer. Swap the Memory Store in by
    calling set_normalized_observation_store() once at the top of a
    notebook/script - no other code changes.
    """
    global _default_store
    if _default_store is None:
        _default_store = LocalNormalizedObservationStore()
    return _default_store


def set_normalized_observation_store(store: NormalizedObservationStore) -> None:
    """Override the default store - this is the swap point for when the
    real Memory Store is ready: set_normalized_observation_store(
    MemoryStoreNormalizedObservationStore(...)) and both agents pick it up
    automatically."""
    global _default_store
    _default_store = store


## 2c. Semantic similarity module (real embeddings, not just word overlap)

**Why this exists:** token-overlap fuzzy matching (e.g. `rapidfuzz.token_set_ratio`) measures shared *words*, not shared *meaning*. Demonstrated failure: `"ATM entrance, main door"` vs `"ATM exit, main door"` scores 81% similar under token overlap despite being close to opposite in meaning. This module uses sentence-embedding cosine similarity (`all-MiniLM-L6-v2`) instead, with automatic graceful fallback to the old lexical approach if the model can't load (no internet, package missing, etc.) - the pipeline never breaks because of this, it just quietly degrades.

**First run downloads the model (~80MB, one-time, ~10-30s)**; cached for the rest of the session.

In [ ]:
"""
ForenSynth-X+ - shared semantic similarity module.

Used by BOTH Entity Resolution and Timeline Agent for location (and,
optionally, general free-text) similarity, replacing pure lexical/token-
overlap matching with actual semantic understanding.

WHY THIS EXISTS (see conversation record / design note):
Token-overlap fuzzy matching (e.g. rapidfuzz.token_set_ratio) measures
shared WORDS, not shared MEANING. Demonstrated failure mode: "ATM entrance,
main door" vs "ATM exit, main door" scores 81% similar under token overlap
despite being close to opposite in meaning - it sees 3 of 4 words shared and
has no notion that "entrance" and "exit" are different concepts. This module
replaces that with sentence-embedding cosine similarity (SentenceTransformer,
all-MiniLM-L6-v2 by default), a standard, citable NLP technique for semantic
textual similarity, chosen over a hand-built synonym/fragment table
specifically because a fragment table only covers vocabulary it was
explicitly given, while embeddings generalize to phrasing never seen before -
a materially stronger claim for a system meant to handle real (not just
generator-bounded) evidence text.

DESIGN: graceful degradation, same pattern as llm_fallback.py. If the model
can't load (package not installed, no network to Hugging Face Hub, etc.),
this silently falls back to the previous lexical fuzzy-matching approach -
the pipeline never blocks or errors because the embedding model isn't
available. Availability and which path was used are exposed so this is
auditable, not hidden.

HONESTY NOTE: the embedding path was built and unit-tested with a mocked
encoder (verifying caching, cosine-similarity math, and fallback wiring are
all correct) but NOT executed end-to-end against the real model in this
development environment, because this sandbox's network egress does not
reach huggingface.co. Verify the actual similarity numbers once in Colab
(which has normal internet access) - see tests/test_semantic_similarity.py
for what's mock-verified vs. what still needs a live check.
"""
from __future__ import annotations

import logging
import re
from typing import Dict, Optional

log = logging.getLogger("forensynth.semantic_similarity")

DEFAULT_MODEL_NAME = "all-MiniLM-L6-v2"

# Fallback lexical scorer (used if the embedding model is unavailable).
try:
    from rapidfuzz import fuzz  # type: ignore
except ImportError:  # pragma: no cover
    fuzz = None


def _lexical_fallback_similarity(a: str, b: str) -> float:
    """Same fallback used elsewhere in the project when rapidfuzz is present,
    with a pure-stdlib Jaccard fallback if it isn't."""
    a, b = (a or "").strip().lower(), (b or "").strip().lower()
    if not a or not b:
        return 0.0
    if fuzz is not None:
        return fuzz.token_set_ratio(a, b) / 100.0
    ta = set(re.findall(r"\w+", a))
    tb = set(re.findall(r"\w+", b))
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


class SemanticSimilarityScorer:
    """
    Lazily loads a sentence-embedding model on first use. Caches one
    embedding per unique string (not per pair) so repeated comparisons
    within a run don't re-encode the same text - important since this gets
    called O(n^2) times over a case's observations.

    Usage:
        scorer = SemanticSimilarityScorer()   # or get_semantic_scorer() below
        score = scorer.similarity("ATM entrance", "ATM entry door")  # in [0, 1]
        scorer.available()   # True if real embeddings are being used
        scorer.backend_used() # "embedding" or "lexical_fallback"
    """

    def __init__(self, model_name: str = DEFAULT_MODEL_NAME) -> None:
        self._model_name = model_name
        self._model = None
        self._ok = False
        self._embedding_cache: Dict[str, "Any"] = {}  # str -> embedding vector
        self._load_attempted = False

    def _ensure_loaded(self) -> None:
        if self._load_attempted:
            return
        self._load_attempted = True
        try:
            from sentence_transformers import SentenceTransformer  # type: ignore
            self._model = SentenceTransformer(self._model_name)
            self._ok = True
            log.info("SemanticSimilarityScorer: loaded '%s'.", self._model_name)
        except Exception as exc:
            log.warning(
                "SemanticSimilarityScorer: could not load embedding model '%s' (%s) - "
                "falling back to lexical fuzzy matching for all similarity scoring. "
                "If this was a transient issue (e.g. a network hiccup), call "
                "retry_load() to try again without restarting the runtime.",
                self._model_name, exc,
            )
            self._model = None
            self._ok = False

    def retry_load(self) -> bool:
        """
        FIX: force a fresh load attempt, clearing any previously cached
        failure. Without this, a single transient failure (e.g. a network
        hiccup while reaching Hugging Face Hub) permanently disables real
        semantic matching for the rest of the session - _ensure_loaded()
        only ever tries once and every later call just returns the cached
        (failed) result, so simply re-running downstream cells does NOT
        retry. Call this explicitly to retry. Returns True if the retry
        succeeded.
        """
        self._load_attempted = False
        self._ok = False
        self._model = None
        self._ensure_loaded()
        return self._ok

    def available(self) -> bool:
        self._ensure_loaded()
        return self._ok

    def backend_used(self) -> str:
        return "embedding" if self.available() else "lexical_fallback"

    def _embed(self, text: str):
        if text in self._embedding_cache:
            return self._embedding_cache[text]
        vec = self._model.encode(text, normalize_embeddings=True)
        self._embedding_cache[text] = vec
        return vec

    def similarity(self, a: Optional[str], b: Optional[str]) -> float:
        """Returns similarity in [0, 1]. Higher = more semantically similar."""
        a, b = (a or "").strip(), (b or "").strip()
        if not a or not b:
            return 0.0
        if a.lower() == b.lower():
            return 1.0

        self._ensure_loaded()
        if not self._ok:
            return _lexical_fallback_similarity(a, b)

        try:
            import numpy as np  # sentence-transformers already depends on this
            ea, eb = self._embed(a.lower()), self._embed(b.lower())
            cos = float(np.dot(ea, eb))  # already normalized -> dot == cosine
            # Cosine can be slightly negative for very dissimilar text;
            # clamp to [0, 1] since callers treat this as a similarity score.
            return max(0.0, min(1.0, cos))
        except Exception as exc:
            log.warning("Embedding similarity call failed (%s) - falling back to lexical for this pair.", exc)
            return _lexical_fallback_similarity(a, b)


_default_scorer: Optional[SemanticSimilarityScorer] = None


def get_semantic_scorer() -> SemanticSimilarityScorer:
    """Module-level singleton so the (potentially slow-to-load) model is
    loaded at most once per process/notebook session, not once per pipeline
    run or per agent instance."""
    global _default_scorer
    if _default_scorer is None:
        _default_scorer = SemanticSimilarityScorer()
    return _default_scorer


def reset_semantic_scorer() -> SemanticSimilarityScorer:
    """
    Reset the module-level singleton and force a fresh load attempt. Call
    this if the model failed to load once (e.g. a transient network issue
    reaching Hugging Face Hub) and you want to retry without restarting the
    whole Colab runtime. Returns the new scorer so you can immediately check
    `.available()`.
    """
    global _default_scorer
    _default_scorer = SemanticSimilarityScorer()
    _default_scorer._ensure_loaded()
    return _default_scorer


def semantic_location_similarity(a: Optional[str], b: Optional[str]) -> float:
    """Convenience wrapper used by both ER and Timeline Agent for location matching."""
    return get_semantic_scorer().similarity(a, b)


## 3. Constants & configuration


In [ ]:
ACTION_TOKENS: Set[str] = {
    "enter", "exit", "withdraw", "deposit", "call", "message",
    "send", "receive", "transfer", "arrive", "depart", "access",
    "attempt", "fail", "succeed", "heading", "headed", "inside", "clear",
}

CONFLICTING_ACTION_PAIRS: List[Tuple[str, str]] = [
    ("enter", "exit"),
    ("withdraw", "deposit"),
    ("arrive", "depart"),
    ("send", "receive"),
    ("succeed", "fail"),
    ("heading", "exited"),
]

WEIGHT_MAP: Dict[str, float] = {
    "entity_coreference":  0.280,
    "mention_consistency": 0.120,
    "temporal":            0.150,
    "location":            0.100,
    "context":             0.180,
    "lexical":             0.050,
    "interaction":         0.070,
    "modality":            0.050,
}
assert abs(sum(WEIGHT_MAP.values()) - 1.0) < 1e-9

CONFIRMED_THRESHOLD:      float = 0.80
CANDIDATE_THRESHOLD_HIGH: float = 0.65
CANDIDATE_THRESHOLD_LOW:  float = 0.50
ATTACHMENT_THRESHOLD:     float = 0.70
CLUSTER_CONFIDENCE_FLOOR: float = 0.55
CROSS_MODAL_MERGE_MIN:    float = 0.58
MERGE_COMPOSITE_MIN:      float = 0.55

OVERSIZED_CLUSTER_FACTOR: float = 3.0
TEMPORAL_WINDOW_SEC:      int   = 300
MAX_TEMPORAL_GAP_SEC:     int   = 3600
MAX_PAIRS:                int   = 500
# FIX: reduced from 40. The chunk that failed needed 6586 tokens against
# a 6000 TPM free-tier cap - i.e. the old size sat right at the edge of
# the limit by design, not as an edge case. 20 pairs/chunk estimates to
# ~3400 tokens (~57% of the cap), giving real margin instead of routinely
# grazing the ceiling.
LLM_BATCH_CHUNK_SIZE:     int   = 20

# FIX: previously each scoring agent chunked ALL candidate pairs at
# 40-per-call with NO ceiling on total API calls - call count scaled
# unboundedly with case size. MAX_LLM_CALLS_PER_RUN is a SHARED budget
# across BOTH agents within one resolve_entities() call (see LLMCallBudget
# below). Once exhausted, remaining chunks silently use the heuristic/
# fuzzy fallback - never blocks, never errors.
# FIX: reduced from 6 to 2 - a hard "at most 1-2 API calls per case
# file" ceiling, shared across BOTH scoring agents combined (not 2 each).
# In practice this typically means the context-scoring agent (which runs
# first in stage 4) gets one real LLM call and the entity-coreference
# agent gets the other; anything beyond that falls back to the heuristic
# scorer. This intentionally trades "most pairs get a real LLM score" for
# "usage is small and predictable" - see the accuracy note below for why
# this doesn't meaningfully cost resolution quality here.
MAX_LLM_CALLS_PER_RUN: int = 2

GROQ_MODEL: str = "llama-3.1-8b-instant"

FEATURE_NAMES: Tuple[str, ...] = (
    "entity_coreference", "mention_consistency", "temporal", "location",
    "context", "lexical", "interaction", "modality",
)

# -- Output classification (CLEAR / PARTIAL / AMBIGUOUS) -----------------------
# Mirrors the Timeline Agent notebook's equivalent classification for
# consistency across both agents.
CLASSIFICATION_CLEAR_MIN_AVG_CONFIDENCE: float = 0.75
CLASSIFICATION_CLEAR_MAX_CONFLICT_FRACTION: float = 0.0
CLASSIFICATION_AMBIGUOUS_MAX_AVG_CONFIDENCE: float = 0.55
CLASSIFICATION_AMBIGUOUS_MIN_CONFLICT_FRACTION: float = 0.30
CLASSIFICATION_AMBIGUOUS_MIN_LOW_CONFIDENCE_FRACTION: float = 0.50

# FIX: constrained re-splitting threshold. When role_count_mismatch fires,
# Stage 10b attempts single-linkage splitting (max spanning tree, cut
# weakest bridging edges). A cut only ever happens if EVERY edge being cut
# scores below this - i.e. there must be a genuinely weak bridge, not just
# the least-strong of several confidently strong bonds. Stricter than
# MERGE_COMPOSITE_MIN on purpose: merging is permissive, splitting is
# destructive.
SPLIT_CUT_MAX_WEIGHT: float = 0.60


## 4. Data classes


In [ ]:
@dataclass
class Observation:
    obs_id:     str
    entity:     str
    role:       str
    modality:   str
    location:   str
    content:    str
    timestamp:  str
    confidence: float
    entity_norm:     str   = ""
    time_offset_sec: int   = 0
    _ts_epoch:       float = field(default=0.0, repr=False)


@dataclass
class HumanConstraints:
    must_merge:     List[Tuple[str, str]]        = field(default_factory=list)
    must_not_merge: List[Tuple[str, str]]        = field(default_factory=list)
    soft_hints:     Dict[Tuple[str, str], float] = field(default_factory=dict)


@dataclass
class PairFeatures:
    obs_a: Observation
    obs_b: Observation
    entity_coreference:  float = 0.0
    mention_consistency: float = 0.0
    temporal:            float = 0.0
    location:            float = 0.0
    context:             float = 0.0
    lexical:             float = 0.0
    interaction:         float = 0.0
    modality:            float = 0.0
    composite:           float = 0.0
    reasons:             List[str] = field(default_factory=list)
    hard_negative:       bool = False

    def compute_composite(self, soft_hint: float = 0.0) -> float:
        raw = sum(getattr(self, name) * WEIGHT_MAP[name] for name in FEATURE_NAMES)
        self.composite = min(1.0, raw + soft_hint)
        return self.composite


@dataclass
class EdgeRecord:
    alias_1:        str
    alias_2:        str
    obs_id_1:       str
    obs_id_2:       str
    weight:         float
    support:        int = 1
    classification: str = "rejected"
    features:       Optional[PairFeatures] = None
    reasons:        List[str] = field(default_factory=list)
    hard_negative:  bool = False


## 5. LLM batch helpers


In [ ]:
# NOTE: normalize_alias() is now defined once in the shared normalization
# cell above (## 2b) and used here and throughout the pipeline - no longer
# duplicated in this cell.


def _parse_llm_json_map(text: str) -> Dict[str, float]:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.MULTILINE)
    text = re.sub(r"```\s*$", "", text, flags=re.MULTILINE).strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{[^{}]*\}", text, flags=re.DOTALL)
        if not match:
            raise
        parsed = json.loads(match.group(0))
    return {str(k): float(v) for k, v in parsed.items() if re.match(r"^\d+$", str(k))}


def _mention_payload(obs: Observation) -> Dict[str, str]:
    return {
        "entity":    obs.entity,
        "role":      obs.role,
        "modality":  obs.modality,
        "location":  obs.location[:120],
        "content":   obs.content[:220],
        "timestamp": obs.timestamp,
    }


class LLMCallBudget:
    """
    Shared, mutable call budget across BOTH scoring agents in one pipeline
    run. Enforces a hard ceiling on total LLM API calls regardless of case
    size - once exhausted, remaining chunks silently use the heuristic
    fallback. Instantiate ONE of these per resolve_entities() call and pass
    it to both ContextScoringAgent and EntityCoreferenceAgent.
    """
    def __init__(self, max_calls: int):
        self.max_calls = max_calls
        self.calls_made = 0

    def try_consume(self) -> bool:
        if self.calls_made >= self.max_calls:
            return False
        self.calls_made += 1
        return True

    @property
    def remaining(self) -> int:
        return max(0, self.max_calls - self.calls_made)

    @property
    def exhausted(self) -> bool:
        return self.calls_made >= self.max_calls


class _BatchLLMScorer:
    """Shared Groq batch caller with chunking, a shared call budget, and local fallback."""

    def __init__(self, model: str, enabled: bool, budget: Optional["LLMCallBudget"] = None):
        self.model = model
        self.enabled = enabled and _GROQ_AVAILABLE
        self._client: Any = None
        self._budget = budget
        if self.enabled:
            api_key = os.environ.get("GROQ_API_KEY", "")
            if api_key:
                self._client = _GroqClient(api_key=api_key)
            else:
                log.warning("GROQ_API_KEY not set – using heuristic/fuzzy fallbacks.")
                self.enabled = False

    def score_batch(
        self,
        payload_items: List[Dict[str, Any]],
        system_prompt: str,
        user_intro: str,
        fallback_fn,
    ) -> List[float]:
        """Return one score per payload item (index-aligned)."""
        n = len(payload_items)
        if n == 0:
            return []

        scores: List[Optional[float]] = [None] * n
        if not self.enabled or self._client is None:
            return [float(fallback_fn(i)) for i in range(n)]

        for chunk_start in range(0, n, LLM_BATCH_CHUNK_SIZE):
            chunk = payload_items[chunk_start : chunk_start + LLM_BATCH_CHUNK_SIZE]

            # FIX: hard budget check before spending an API call. Once the
            # shared budget is exhausted, every remaining chunk (across BOTH
            # agents, for the rest of this pipeline run) uses the fallback
            # scorer instead.
            if self._budget is not None and not self._budget.try_consume():
                log.info("LLM call budget exhausted (%d/%d used) - falling back to heuristic scoring for remaining %d pair(s).",
                          self._budget.calls_made, self._budget.max_calls, len(chunk))
                for local_i in range(len(chunk)):
                    scores[chunk_start + local_i] = float(fallback_fn(chunk_start + local_i))
                continue

            prompt = (
                f"{system_prompt}\n\n{user_intro}\n"
                + json.dumps(chunk, ensure_ascii=False)
                + '\n\nReturn ONLY JSON: {"0": 0.85, "1": 0.12, ...} with one float 0.0-1.0 per id.'
            )
            max_tok = min(4096, max(256, len(chunk) * 16 + 128))
            raw_scores: Dict[str, float] = {}
            try:
                resp = self._client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=max_tok,
                    temperature=0.0,
                )
                raw_scores = _parse_llm_json_map(resp.choices[0].message.content or "")
            except Exception as exc:
                log.warning("LLM batch chunk failed (%s); using fallback for %d pairs.", exc, len(chunk))

            for local_i in range(len(chunk)):
                global_i = chunk_start + local_i
                if str(local_i) in raw_scores:
                    scores[global_i] = max(0.0, min(1.0, raw_scores[str(local_i)]))
                else:
                    scores[global_i] = float(fallback_fn(global_i))

        return [s if s is not None else float(fallback_fn(i)) for i, s in enumerate(scores)]


## 6. Context scoring agent (LLM)


In [ ]:
class ContextScoringAgent:
  def __init__(self, model: str = GROQ_MODEL, enabled: bool = True, budget: Optional["LLMCallBudget"] = None):
    self._llm = _BatchLLMScorer(model, enabled, budget=budget)
    self._cache: Dict[Tuple[str, str], float] = {}

  @staticmethod
  def _cache_key(a: str, b: str) -> Tuple[str, str]:
    ka, kb = a[:120], b[:120]
    return (ka, kb) if ka <= kb else (kb, ka)

  def _detect_conflicting_actions(self, text_a: str, text_b: str) -> float:
    def _tokens(t: str) -> Set[str]:
      return ACTION_TOKENS & set(re.findall(r"\b\w+\b", t.lower()))
    ta, tb = _tokens(text_a), _tokens(text_b)
    for act_a, act_b in CONFLICTING_ACTION_PAIRS:
      if (act_a in ta and act_b in tb) or (act_b in ta and act_a in tb):
        return 0.5
    return 1.0

  def _content_fallback(self, pair: Tuple[str, str]) -> float:
    ca, cb = pair
    penalty = self._detect_conflicting_actions(ca, cb)
    return min(1.0, max(0.0, fuzz.token_set_ratio(ca, cb) / 100.0 * penalty))

  def precompute_batch_scores(
    self,
    candidate_pairs: List[Tuple[str, str]],
    obs_by_id: Dict[str, Observation],
  ) -> None:
    unique: List[Tuple[str, str]] = []
    seen: Set[Tuple[str, str]] = set()
    for id_a, id_b in candidate_pairs:
      oa, ob = obs_by_id.get(id_a), obs_by_id.get(id_b)
      if not oa or not ob:
        continue
      ck = self._cache_key(oa.content, ob.content)
      if ck in self._cache or ck in seen:
        continue
      seen.add(ck)
      unique.append(ck)

    if not unique:
      return

    log.info("Context precompute: %d unique content pairs.", len(unique))
    system = (
      "You are a forensic semantic analyst. Score how likely two observation texts "
      "describe the same underlying event or the same actor's actions."
    )
    intro = "Rate each text pair from 0.0 (unrelated) to 1.0 (same event/entity)."
    payload = [
      {"id": str(i), "text_a": a[:300], "text_b": b[:300]}
      for i, (a, b) in enumerate(unique)
    ]

    def fallback(i: int) -> float:
      a, b = unique[i]
      return self._content_fallback((a, b))

    scored = self._llm.score_batch(payload, system, intro, fallback)
    for pair_key, raw, (a, b) in zip(unique, scored, unique):
      penalty = self._detect_conflicting_actions(a, b)
      self._cache[pair_key] = min(1.0, max(0.0, raw * penalty))

  def score(self, content_a: str, content_b: str) -> float:
    ck = self._cache_key(content_a, content_b)
    if ck not in self._cache:
      self._cache[ck] = self._content_fallback(ck)
    return self._cache[ck]


## 7. Entity co-reference agent (LLM)


In [ ]:
class EntityCoreferenceAgent:
  """LLM scores whether two forensic mentions refer to the same real-world entity."""

  def __init__(self, model: str = GROQ_MODEL, enabled: bool = True, budget: Optional["LLMCallBudget"] = None):
    self._llm = _BatchLLMScorer(model, enabled, budget=budget)
    self._cache: Dict[Tuple[str, str], float] = {}

  @staticmethod
  def _pair_key(id_a: str, id_b: str) -> Tuple[str, str]:
    return (id_a, id_b) if id_a <= id_b else (id_b, id_a)

  @staticmethod
  def heuristic_coreference(oa: Observation, ob: Observation, temporal_window: int, max_gap: int) -> float:
    if oa.entity_norm == ob.entity_norm:
      return 1.0
    if oa.role.strip().lower() != ob.role.strip().lower():
      return 0.12

    dt = abs(oa.time_offset_sec - ob.time_offset_sec)
    if dt > max_gap:
      return 0.05

    cross_modal = oa.modality.lower() != ob.modality.lower()
    score = 0.45 if cross_modal else 0.32

    if dt <= temporal_window:
      score += 0.25
    elif dt <= max_gap:
      score += 0.10

    la, lb = oa.location.strip().lower(), ob.location.strip().lower()
    if la and lb:
      loc_sim = fuzz.token_set_ratio(la, lb) / 100.0
      if loc_sim >= 0.45:
        score += 0.10
      shared_tokens = set(re.findall(r"[a-z0-9]+", la)) & set(re.findall(r"[a-z0-9]+", lb))
      if shared_tokens & {"atm", "server", "booth", "room", "network", "ssh", "login"}:
        score += 0.08

    content_sim = fuzz.token_set_ratio(oa.content, ob.content) / 100.0
    if content_sim >= 0.35:
      score += 0.08

    topical = {"enter", "entered", "inside", "heading", "headed", "starting", "exit", "exited",
               "coming", "leaving", "depart", "server", "ssh", "login", "access", "atm", "booth", "clear"}
    ca = set(re.findall(r"[a-z0-9]+", oa.content.lower()))
    cb = set(re.findall(r"[a-z0-9]+", ob.content.lower()))
    overlap = ca & cb & topical
    if overlap:
      score += min(0.14, 0.05 * len(overlap))

    enter_tokens = {"enter", "entered", "inside", "heading", "headed", "starting"}
    exit_tokens = {"exit", "exited", "coming", "leaving", "depart"}
    a_enter = bool(ca & enter_tokens)
    b_enter = bool(cb & enter_tokens)
    a_exit = bool(ca & exit_tokens)
    b_exit = bool(cb & exit_tokens)
    if (a_enter and b_enter) or (a_exit and b_exit):
      score += 0.10

    # Narrative phase bucketing: approach / inside / exit within one event window
    def _phase(tokens: Set[str]) -> str:
      if tokens & exit_tokens:
        return "exit"
      if tokens & enter_tokens:
        return "inside"
      if tokens & {"heading", "headed", "towards", "see", "activity"}:
        return "approach"
      return "other"

    if dt <= temporal_window and _phase(ca) == _phase(cb) != "other":
      score += 0.08

    return min(1.0, score)

  def precompute_batch_scores(
    self,
    candidate_pairs: List[Tuple[str, str]],
    obs_by_id: Dict[str, Observation],
    temporal_window: int,
    max_gap: int,
  ) -> None:
    unique_keys: List[Tuple[str, str]] = []
    seen: Set[Tuple[str, str]] = set()
    for id_a, id_b in candidate_pairs:
      key = self._pair_key(id_a, id_b)
      if key in self._cache or key in seen:
        continue
      seen.add(key)
      unique_keys.append(key)

    if not unique_keys:
      return

    log.info("Entity coreference precompute: %d unique mention pairs.", len(unique_keys))
    obs_pairs = [(obs_by_id[a], obs_by_id[b]) for a, b in unique_keys]
    payload = [
      {"id": str(i), "mention_a": _mention_payload(oa), "mention_b": _mention_payload(ob)}
      for i, (oa, ob) in enumerate(obs_pairs)
    ]

    system = (
      "You are a digital forensics entity-resolution expert. Decide if mention A and "
      "mention B refer to the SAME real-world person/device/account, even when names differ "
      "(e.g. 'Suspect A', an IP, and 'person in red jacket'). Use role, modality, location, "
      "timestamp, and content together. Different roles (suspect vs witness) should score low."
    )
    intro = "Score each mention pair from 0.0 (definitely different) to 1.0 (same entity)."

    def fallback(i: int) -> float:
      oa, ob = obs_pairs[i]
      return self.heuristic_coreference(oa, ob, temporal_window, max_gap)

    scored = self._llm.score_batch(payload, system, intro, fallback)
    for i, (key, raw) in enumerate(zip(unique_keys, scored)):
      # FIX (real production regression): the LLM path used to fully
      # REPLACE the heuristic score whenever available, instead of
      # supplementing it. Observed effect on real CASE_ATM_001 data:
      # three cross-modal same-witness pairs heuristic_coreference
      # correctly scored 0.65-0.69 (merged) dropped to 0.42-0.48
      # (rejected) once the LLM was live - the LLM read one
      # observations literal content ("...leave the office" - a
      # wording artifact, not the ATM) and was more conservative than
      # the heuristics role/temporal/topical-keyword logic. Entity
      # count went from the correct 2 to a fragmented 4. Blending via
      # max() means the LLM can still ADD confidence beyond the
      # heuristic, but can never subtract from it.
      heuristic_score = fallback(i)
      self._cache[key] = max(0.0, min(1.0, max(raw, heuristic_score)))

  def score(self, oa: Observation, ob: Observation, temporal_window: int, max_gap: int) -> float:
    key = self._pair_key(oa.obs_id, ob.obs_id)
    if key not in self._cache:
      self._cache[key] = self.heuristic_coreference(oa, ob, temporal_window, max_gap)
    return self._cache[key]


## 8. Union-Find clustering


In [ ]:
class UnionFind:
  def __init__(self, elements: List[str]):
    self.parent = {e: e for e in elements}
    self.rank = {e: 0 for e in elements}

  def find(self, x: str) -> str:
    while self.parent[x] != x:
      self.parent[x] = self.parent[self.parent[x]]
      x = self.parent[x]
    return x

  def union(self, x: str, y: str) -> bool:
    px, py = self.find(x), self.find(y)
    if px == py:
      return False
    if self.rank[px] < self.rank[py]:
      px, py = py, px
    self.parent[py] = px
    if self.rank[px] == self.rank[py]:
      self.rank[px] += 1
    return True

  def groups(self) -> Dict[str, List[str]]:
    d: Dict[str, List[str]] = defaultdict(list)
    for e in self.parent:
      d[self.find(e)].append(e)
    return dict(d)


## 8b. Output classification (CLEAR / PARTIAL / AMBIGUOUS)

Mirrors the Timeline Agent notebook's equivalent classification, added for consistency across both agents.

In [ ]:
def classify_resolution_output(canonical_entities, conflicts):
    """
    Signal-based CLEAR / PARTIAL / AMBIGUOUS classification of this
    resolution run's own confidence, mirroring the Timeline Agent
    notebook's equivalent classification for consistency across both
    agents. Honest scope note: classifies THIS run's confidence using
    concrete already-computed signals - not a comparison across
    alternative resolution hypotheses.
    """
    if not canonical_entities:
        return "AMBIGUOUS", "No canonical entities were resolved."

    confidences = [e.get("confidence_score", 0.0) for e in canonical_entities]
    avg_confidence = sum(confidences) / len(confidences)
    low_confidence_fraction = sum(1 for c in confidences if c < CLUSTER_CONFIDENCE_FLOOR) / len(confidences)
    conflict_fraction = len(conflicts) / len(canonical_entities)

    is_ambiguous = (
        avg_confidence <= CLASSIFICATION_AMBIGUOUS_MAX_AVG_CONFIDENCE
        or conflict_fraction >= CLASSIFICATION_AMBIGUOUS_MIN_CONFLICT_FRACTION
        or low_confidence_fraction >= CLASSIFICATION_AMBIGUOUS_MIN_LOW_CONFIDENCE_FRACTION
    )
    if is_ambiguous:
        return (
            "AMBIGUOUS",
            f"avg_confidence={avg_confidence:.2f}, conflict_fraction={conflict_fraction:.2f}, "
            f"low_confidence_fraction={low_confidence_fraction:.2f} - at least one signal crossed "
            "the ambiguity threshold; treat these entity mappings as low-trust and prioritise "
            "investigator review before downstream use.",
        )

    if avg_confidence >= CLASSIFICATION_CLEAR_MIN_AVG_CONFIDENCE and conflict_fraction <= CLASSIFICATION_CLEAR_MAX_CONFLICT_FRACTION:
        return (
            "CLEAR",
            f"avg_confidence={avg_confidence:.2f}, no conflicts detected - entity resolution is well-supported.",
        )

    return (
        "PARTIAL",
        f"avg_confidence={avg_confidence:.2f}, conflict_fraction={conflict_fraction:.2f}, "
        f"low_confidence_fraction={low_confidence_fraction:.2f} - resolution is usable but has "
        "some flagged or low-confidence clusters worth investigator attention.",
    )


## 9. Entity resolution pipeline (12 stages)


In [ ]:
class EntityResolutionPipeline:
  def __init__(
    self,
    config: Optional[Dict[str, Any]] = None,
    human_constraints: Optional[HumanConstraints] = None,
    llm_enabled: bool = True,
    max_llm_calls: int = MAX_LLM_CALLS_PER_RUN,
  ):
    cfg = config or {}
    self.check_duplicates = cfg.get("check_duplicates", True)
    self.case_base_time = cfg.get("case_base_time", None)
    self.temporal_window_sec = cfg.get("temporal_window_sec", TEMPORAL_WINDOW_SEC)
    self.max_temporal_gap_sec = cfg.get("max_temporal_gap_sec", MAX_TEMPORAL_GAP_SEC)
    self.max_pairs = cfg.get("max_pairs", MAX_PAIRS)
    self.confirmed_threshold = cfg.get("confirmed_threshold", CONFIRMED_THRESHOLD)
    self.candidate_threshold_low = cfg.get("candidate_threshold_low", CANDIDATE_THRESHOLD_LOW)
    self.candidate_threshold_high = cfg.get("candidate_threshold_high", CANDIDATE_THRESHOLD_HIGH)
    self.llm_enabled = llm_enabled

    self.constraints = human_constraints or HumanConstraints()
    # FIX: previously one shared LLMCallBudget across both agents meant
    # whichever agent ran first (context-scoring) could consume the ENTIRE
    # budget, leaving entity-coreference - arguably the more consequential
    # signal for merge decisions - with zero real LLM calls on anything
    # but the smallest cases. Each agent now gets its own guaranteed
    # share (ceil/floor split of max_llm_calls) instead of competing for
    # one pool, so both signal sources get at least one real LLM-scored
    # chunk whenever max_llm_calls >= 2.
    context_budget_size = (max_llm_calls + 1) // 2
    entity_budget_size = max_llm_calls // 2
    self._context_budget = LLMCallBudget(context_budget_size)
    self._entity_budget = LLMCallBudget(entity_budget_size)
    self.context_agent = ContextScoringAgent(model=GROQ_MODEL, enabled=self.llm_enabled, budget=self._context_budget)
    self.entity_agent = EntityCoreferenceAgent(model=GROQ_MODEL, enabled=self.llm_enabled, budget=self._entity_budget)

    self.case_id = "UNKNOWN_CASE"
    self.observations: List[Observation] = []
    self.base_epoch = 0.0
    self.candidate_pairs: List[Tuple[str, str]] = []
    self.edges: List[EdgeRecord] = []
    self.graph = nx.Graph()
    self.clusters: Dict[str, List[str]] = {}
    self.conflicts: List[Dict[str, Any]] = []
    self.fir_role_counts: Dict[str, int] = {}
    self._resplit_log: List[Dict[str, Any]] = []
    self.status = "success"
    self.error_message = ""
    self.stage_timings: Dict[str, float] = {}
    self._obs_by_id: Dict[str, Observation] = {}
    self._pair_features: Dict[Tuple[str, str], PairFeatures] = {}
    self._candidate_data: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    self._canonical_entities: List[Dict[str, Any]] = []

  def run(self, raw_input: Dict[str, Any]) -> Dict[str, Any]:
    pipeline_start = time.perf_counter()
    try:
      self._stage(1, self._stage_1_intake, raw_input)
      self._stage(2, self._stage_2_normalization)
      self._stage(3, self._stage_3_blocking)
      self._stage(4, self._stage_4_feature_computation)
      self._stage(5, self._stage_5_scoring)
      self._stage(6, self._stage_6_classification)
      self._stage(7, self._stage_7_graph_construction)
      self._stage(8, self._stage_8_clustering)
      self._stage(9, self._stage_9_guarded_attachment)
      self._stage(10, self._stage_10_conflict_detection)
      self._stage("10b", self._stage_10b_constrained_resplit)
      self._stage(11, self._stage_11_entity_labeling)
    except Exception as exc:
      log.exception("Pipeline failed: %s", exc)
      self.status = "failed"
      self.error_message = str(exc)
    return self._stage_12_package(time.perf_counter() - pipeline_start)

  def _stage(self, n: Any, fn: Any, *args: Any) -> None:
    t0 = time.perf_counter()
    fn(*args)
    suffix = fn.__name__.split("_stage_")[1]
    self.stage_timings[f"stage_{suffix}"] = time.perf_counter() - t0
    log.info("Stage %s (%s) done in %.4fs", n, fn.__name__, self.stage_timings[f"stage_{suffix}"])

  @staticmethod
  def _canonical_pair(a: str, b: str) -> Tuple[str, str]:
    return (min(a, b), max(a, b))

  @staticmethod
  def _parse_ts(ts_str: str) -> float:
    # FIX: delegates to the shared parse_timestamp() (## 2b) instead of
    # maintaining its own separate 4-format list - this is what previously
    # let ER and the Timeline Agent silently disagree on which timestamps
    # were parseable.
    return parse_timestamp(ts_str)

  def _stage_1_intake(self, raw_input: Dict[str, Any]) -> None:
    self.case_id = str(raw_input.get("case_id", "") or "UNKNOWN_CASE")
    # FIR-declared role counts are legitimate input data (the complaint
    # itself), not derived from ground truth - captured so Stage 10 can
    # flag a mismatch against resolved entity counts.
    self.fir_role_counts: Dict[str, int] = dict(raw_input.get("fir", {}).get("roles", {}) or {})
    raw_obs = raw_input.get("observations", [])

    # FIX: normalization (role/modality canonicalization, content cleaning,
    # timestamp parsing, entity_norm, time_offset_sec) now goes through the
    # single shared batch entry point (normalize_case(), via the store - see
    # ## 2b) instead of being computed inline here AND again in
    # _stage_2_normalization. This is the same seam the Timeline Agent
    # notebook now uses - the swap point for the real Memory Store later:
    # once the team's store is ready, call set_normalized_observation_store(
    # MemoryStoreNormalizedObservationStore(...)) once and this intake code
    # does not need to change.
    store = get_normalized_observation_store()
    try:
      normalized_list = store.get(self.case_id, raw_obs)
    except Exception as exc:
      log.error("Normalization store failed for case '%s' (%s) - falling back to empty observation set.", self.case_id, exc)
      normalized_list = [None] * len(raw_obs)

    seen_hashes: Dict[str, Observation] = {}
    for item, n in zip(raw_obs, normalized_list):
      obs_id = str(item.get("obs_id", "")).strip()
      entity = str(item.get("entity", "")).strip()
      if n is None or not obs_id or not entity:
        log.warning("Skipping malformed obs %s: missing obs_id/entity or normalization failed", item.get("obs_id"))
        continue
      obs = Observation(
        obs_id=n.obs_id,
        entity=n.entity_raw,
        role=n.role,
        modality=n.modality,
        location=n.location_raw,
        content=n.content_clean,
        timestamp=n.timestamp_raw,
        confidence=n.confidence,
        entity_norm=n.entity_norm,
        time_offset_sec=n.time_offset_sec,
        _ts_epoch=n.ts_epoch,
      )
      if self.check_duplicates:
        fp = f"{obs.entity}|{obs.modality}|{obs.location}|{obs.content}|{obs.timestamp}"
        digest = hashlib.sha256(fp.encode()).hexdigest()
        if digest not in seen_hashes or obs.confidence > seen_hashes[digest].confidence:
          seen_hashes[digest] = obs
      else:
        seen_hashes[obs.obs_id] = obs
    self.observations = list(seen_hashes.values())
    self._obs_by_id = {o.obs_id: o for o in self.observations}

  def _stage_2_normalization(self) -> None:
    # FIX: ts_epoch / time_offset_sec / entity_norm are now already
    # populated by normalize_case() in Stage 1 (single normalization
    # boundary, computed once) - this stage is intentionally a no-op, kept
    # only so stage numbering and stage_timings output stay stable.
    pass

  def _build_obs_blocked_set(self) -> Set[Tuple[str, str]]:
    blocked: Set[Tuple[str, str]] = set()
    for a_alias, b_alias in self.constraints.must_not_merge:
      a_norm, b_norm = normalize_alias(a_alias), normalize_alias(b_alias)
      for oa in self.observations:
        for ob in self.observations:
          if oa.obs_id == ob.obs_id:
            continue
          if {oa.entity_norm, ob.entity_norm} == {a_norm, b_norm}:
            blocked.add(self._canonical_pair(oa.obs_id, ob.obs_id))
    return blocked

  def _stage_3_blocking(self) -> None:
    blocked = self._build_obs_blocked_set()
    forced: Set[Tuple[str, str]] = set()
    for a_alias, b_alias in self.constraints.must_merge:
      a_norm, b_norm = normalize_alias(a_alias), normalize_alias(b_alias)
      ids_a = [o.obs_id for o in self.observations if o.entity_norm == a_norm]
      ids_b = [o.obs_id for o in self.observations if o.entity_norm == b_norm]
      for ia, ib in itertools.product(ids_a, ids_b):
        if ia != ib:
          key = self._canonical_pair(ia, ib)
          if key not in blocked:
            forced.add(key)

    candidate_set: Set[Tuple[str, str]] = set()
    obs_list = self.observations
    for i, oa in enumerate(obs_list):
      for ob in obs_list[i + 1 :]:
        key = self._canonical_pair(oa.obs_id, ob.obs_id)
        if key in blocked:
          continue
        dt = abs(oa.time_offset_sec - ob.time_offset_sec)
        same_loc = bool(oa.location.strip()) and oa.location.strip().lower() == ob.location.strip().lower()
        same_role = oa.role.strip().lower() == ob.role.strip().lower()
        cross_modal = oa.modality.lower() != ob.modality.lower()
        if same_loc or dt <= self.temporal_window_sec or (cross_modal and same_role and dt <= self.max_temporal_gap_sec):
          candidate_set.add(key)

    all_candidates = list(forced) + [p for p in candidate_set if p not in forced]
    self.candidate_pairs = all_candidates[: self.max_pairs]

  def _feat_mention_consistency(self, oa: Observation, ob: Observation) -> Tuple[float, bool]:
    if oa.modality == ob.modality:
      if oa.entity_norm == ob.entity_norm:
        dt = abs(oa.time_offset_sec - ob.time_offset_sec)
        loc = oa.location.strip().lower() == ob.location.strip().lower() if oa.location and ob.location else False
        if dt <= self.temporal_window_sec and loc:
          return 0.85, True
        if dt <= self.max_temporal_gap_sec:
          return 0.45, False
        return 0.20, False
      return 0.30, False
    dt = abs(oa.time_offset_sec - ob.time_offset_sec)
    if dt <= self.temporal_window_sec:
      return 0.78, True
    if dt <= self.max_temporal_gap_sec:
      return 0.42, False
    return 0.10, False

  def _feat_temporal(self, oa: Observation, ob: Observation) -> float:
    dt = abs(oa.time_offset_sec - ob.time_offset_sec)
    if dt > self.max_temporal_gap_sec:
      return 0.0
    if dt <= self.temporal_window_sec:
      return max(0.0, 1.0 - dt / self.temporal_window_sec)
    span = self.max_temporal_gap_sec - self.temporal_window_sec
    return max(0.0, 0.3 * (1.0 - (dt - self.temporal_window_sec) / span))

  def _feat_location(self, oa: Observation, ob: Observation) -> float:
    la, lb = oa.location.strip().lower(), ob.location.strip().lower()
    if not la or not lb:
      return 0.35
    if la == lb:
      return 1.0
    # FIX: semantic similarity instead of pure token overlap - see ## 2c.
    # Old approach scored "ATM entrance" vs "ATM exit" as 81% similar
    # (shared words "ATM"/"main"/"door"), despite opposite meaning.
    return semantic_location_similarity(la, lb)

  def _feat_context(self, oa: Observation, ob: Observation) -> float:
    return self.context_agent.score(oa.content, ob.content)

  def _feat_lexical(self, oa: Observation, ob: Observation) -> float:
    return fuzz.token_sort_ratio(oa.entity_norm, ob.entity_norm) / 100.0

  def _feat_interaction(self, oa: Observation, ob: Observation) -> float:
    modality_pairs = {
      frozenset({"video", "audio"}): 0.85,
      frozenset({"video", "text"}): 0.72,
      frozenset({"audio", "text"}): 0.68,
      frozenset({"video"}): 0.50,
      frozenset({"audio"}): 0.50,
      frozenset({"text"}): 0.45,
    }
    base = modality_pairs.get(frozenset({oa.modality.lower(), ob.modality.lower()}), 0.30)
    if oa.role and ob.role and oa.role.lower() != ob.role.lower():
      conflicting = {frozenset({"suspect", "witness"}), frozenset({"suspect", "victim"}), frozenset({"perpetrator", "victim"})}
      if frozenset({oa.role.lower(), ob.role.lower()}) in conflicting:
        base *= 0.55
    return min(1.0, base)

  def _feat_modality(self, oa: Observation, ob: Observation) -> float:
    return 0.40 if oa.modality.lower() == ob.modality.lower() else 0.82

  def _feat_entity_coreference(self, oa: Observation, ob: Observation) -> float:
    # FIX: an exact alias match is near-certain evidence of the same entity
    # (the generator guarantees no alias string is reused across different
    # real entities within a modality). The heuristic fallback already
    # encoded this (returns 1.0 for oa.entity_norm == ob.entity_norm), but
    # the LLM path did not enforce it - it scores pairs holistically on
    # content, and can under-score two same-alias observations describing
    # visibly different moments (e.g. two different actions by the same
    # witness), pulling the merge composite below threshold and splitting
    # one real entity into two clusters. This override makes the guarantee
    # hold regardless of which scoring path (LLM or heuristic) is active.
    if oa.entity_norm == ob.entity_norm:
        return 1.0
    return self.entity_agent.score(oa, ob, self.temporal_window_sec, self.max_temporal_gap_sec)

  def _stage_4_feature_computation(self) -> None:
    self.context_agent.precompute_batch_scores(self.candidate_pairs, self._obs_by_id)
    self.entity_agent.precompute_batch_scores(
      self.candidate_pairs, self._obs_by_id, self.temporal_window_sec, self.max_temporal_gap_sec
    )
    self._pair_features = {}
    for key in self.candidate_pairs:
      oa, ob = self._obs_by_id.get(key[0]), self._obs_by_id.get(key[1])
      if not oa or not ob:
        continue
      pf = PairFeatures(obs_a=oa, obs_b=ob)
      pf.entity_coreference = self._feat_entity_coreference(oa, ob)
      pf.mention_consistency, _ = self._feat_mention_consistency(oa, ob)
      pf.temporal = self._feat_temporal(oa, ob)
      pf.location = self._feat_location(oa, ob)
      pf.context = self._feat_context(oa, ob)
      pf.lexical = self._feat_lexical(oa, ob)
      pf.interaction = self._feat_interaction(oa, ob)
      pf.modality = self._feat_modality(oa, ob)
      self._pair_features[key] = pf

  def _stage_5_scoring(self) -> None:
    blocked_alias_pairs = {
      self._canonical_pair(normalize_alias(a), normalize_alias(b))
      for a, b in self.constraints.must_not_merge
    }
    for key, pf in self._pair_features.items():
      oa, ob = pf.obs_a, pf.obs_b
      hint_key = self._canonical_pair(oa.entity_norm, ob.entity_norm)
      pf.compute_composite(soft_hint=self.constraints.soft_hints.get(hint_key, 0.0))
      pf.reasons = [name for name in FEATURE_NAMES if getattr(pf, name) > 0.5]
      if self._canonical_pair(oa.entity_norm, ob.entity_norm) in blocked_alias_pairs:
        pf.composite = 0.0
        pf.reasons = []
        pf.hard_negative = True
      # FIX: real over-merging bug, verified against CASE_ATM_002 ground
      # truth. The generator guarantees each real entity gets AT MOST
      # ONE alias per modality (video Person_NN, audio Speaker_X, text
      # from a fixed set - never two different alias strings for the
      # same entity in one modality). The converse is a hard, provable
      # fact: two DIFFERENT alias strings in the SAME modality can never
      # be the same real entity. Without this, the cross-modal bonus
      # (designed to bridge one person across sensors) incorrectly also
      # bridges multiple coordinating same-role actors on different
      # channels - verified: Person_97/Person_50 (both video),
      # Speaker_D/Speaker_Q (both audio) were being merged despite
      # being different real suspects, purely because this invariant was
      # never checked.
      elif oa.modality == ob.modality and oa.entity_norm != ob.entity_norm:
        pf.composite = 0.0
        pf.reasons = []
        pf.hard_negative = True

  def _stage_6_classification(self) -> None:
    self.edges = []
    for pf in self._pair_features.values():
      if pf.hard_negative:
        cls = "rejected"
      elif pf.composite >= self.confirmed_threshold:
        cls = "confirmed"
      elif pf.composite >= self.candidate_threshold_high:
        cls = "likely"
      elif pf.composite >= self.candidate_threshold_low:
        cls = "possible"
      else:
        cls = "rejected"
      self.edges.append(
        EdgeRecord(
          alias_1=pf.obs_a.entity_norm,
          alias_2=pf.obs_b.entity_norm,
          obs_id_1=pf.obs_a.obs_id,
          obs_id_2=pf.obs_b.obs_id,
          weight=pf.composite,
          classification=cls,
          features=pf,
          reasons=list(pf.reasons),
          hard_negative=pf.hard_negative,
        )
      )

  def _stage_7_graph_construction(self) -> None:
    G = nx.Graph()
    for obs in self.observations:
      G.add_node(obs.obs_id)
    for edge in self.edges:
      if edge.classification != "confirmed":
        continue
      if G.has_edge(edge.obs_id_1, edge.obs_id_2):
        G[edge.obs_id_1][edge.obs_id_2]["weight"] = max(G[edge.obs_id_1][edge.obs_id_2]["weight"], edge.weight)
        G[edge.obs_id_1][edge.obs_id_2]["support"] += 1
      else:
        G.add_edge(edge.obs_id_1, edge.obs_id_2, weight=edge.weight, support=1)
    self.graph = G

  def _stage_8_clustering(self) -> None:
    components = {f"C{i+1}": sorted(comp) for i, comp in enumerate(nx.connected_components(self.graph))}
    uf = UnionFind([o.obs_id for o in self.observations])
    for comp in components.values():
      for j in range(1, len(comp)):
        uf.union(comp[0], comp[j])

    # FIX: unconditional same-alias merge pass, decoupled entirely from
    # edge classification/composite scoring. Without this, a same-alias
    # pair whose OTHER features (context, location, ...) score low enough
    # to get classified "rejected" (composite < CANDIDATE_THRESHOLD_LOW)
    # never even reaches the same-alias merge logic below, regardless of
    # the _feat_entity_coreference override - splitting one real entity
    # into two clusters purely because content happened to read as
    # dissimilar. Same alias string within one modality is near-certain
    # ground truth (the generator guarantees no alias reuse across real
    # entities), so this merges unconditionally, gated only by a generous
    # temporal sanity bound - never by composite score or edge
    # classification, so it cannot be undermined by LLM scoring variance.
    by_alias: Dict[str, List[Observation]] = defaultdict(list)
    for obs in self.observations:
      by_alias[obs.entity_norm].append(obs)
    for alias, obs_group in by_alias.items():
      if len(obs_group) < 2:
        continue
      base = obs_group[0]
      for other in obs_group[1:]:
        dt = abs(base.time_offset_sec - other.time_offset_sec)
        if dt <= self.max_temporal_gap_sec:
          uf.union(base.obs_id, other.obs_id)

    for edge in self.edges:
      if edge.hard_negative or edge.classification == "rejected":
        continue
      oa, ob = self._obs_by_id[edge.obs_id_1], self._obs_by_id[edge.obs_id_2]
      dt = abs(oa.time_offset_sec - ob.time_offset_sec)
      temporal_ok = dt <= self.max_temporal_gap_sec
      role_ok = oa.role.strip().lower() == ob.role.strip().lower()
      entity_corr = edge.features.entity_coreference if edge.features else 0.0
      composite = edge.weight

      if oa.entity_norm == ob.entity_norm:
        # Same alias can move across locations during an incident.
        if temporal_ok and composite >= MERGE_COMPOSITE_MIN:
          uf.union(edge.obs_id_1, edge.obs_id_2)
      elif (
        role_ok
        and temporal_ok
        and entity_corr >= CROSS_MODAL_MERGE_MIN
        and composite >= MERGE_COMPOSITE_MIN
      ):
        uf.union(edge.obs_id_1, edge.obs_id_2)

    self.clusters = {f"C{i+1}": sorted(m) for i, (_, m) in enumerate(uf.groups().items())}

  def _stage_9_guarded_attachment(self) -> None:
    obs_to_cluster = {oid: cid for cid, members in self.clusters.items() for oid in members}
    self._candidate_data = defaultdict(list)

    def is_singleton(obs_id: str) -> bool:
      cid = obs_to_cluster.get(obs_id)
      return cid is not None and len(self.clusters.get(cid, [])) == 1

    for edge in self.edges:
      if edge.classification not in ("likely", "possible") or edge.hard_negative:
        continue
      for singleton_id, partner_id in (
        (edge.obs_id_1, edge.obs_id_2) if is_singleton(edge.obs_id_1) else (None, None),
        (edge.obs_id_2, edge.obs_id_1) if is_singleton(edge.obs_id_2) else (None, None),
      ):
        if not singleton_id:
          continue
        partner_cid = obs_to_cluster.get(partner_id)
        if not partner_cid:
          continue
        if edge.weight >= ATTACHMENT_THRESHOLD:
          old_cid = obs_to_cluster[singleton_id]
          self.clusters[partner_cid].append(singleton_id)
          self.clusters[old_cid].remove(singleton_id)
          if not self.clusters[old_cid]:
            del self.clusters[old_cid]
          obs_to_cluster[singleton_id] = partner_cid
        else:
          s_obs, p_obs = self._obs_by_id[singleton_id], self._obs_by_id[partner_id]
          reasons = list(edge.reasons)
          self._candidate_data[singleton_id].append({"candidate_alias": p_obs.entity_norm, "score": round(edge.weight, 4), "reasons": reasons})
          self._candidate_data[partner_id].append({"candidate_alias": s_obs.entity_norm, "score": round(edge.weight, 4), "reasons": reasons})
    self.clusters = {k: v for k, v in self.clusters.items() if v}

  def _cluster_confidence(self, cluster_obs_ids: List[str]) -> float:
    if len(cluster_obs_ids) <= 1:
      obs = self._obs_by_id.get(cluster_obs_ids[0]) if cluster_obs_ids else None
      return obs.confidence if obs else 0.5
    member_set = set(cluster_obs_ids)
    total_weight = weighted_sum = 0.0
    for edge in self.edges:
      if edge.classification == "confirmed" and edge.obs_id_1 in member_set and edge.obs_id_2 in member_set:
        oa, ob = self._obs_by_id[edge.obs_id_1], self._obs_by_id[edge.obs_id_2]
        w = (oa.confidence + ob.confidence) / 2.0
        total_weight += w
        weighted_sum += w * edge.weight
    if total_weight == 0.0:
      confs = [self._obs_by_id[oid].confidence for oid in cluster_obs_ids if oid in self._obs_by_id]
      return sum(confs) / len(confs) if confs else 0.5
    return weighted_sum / total_weight

  def _stage_10_conflict_detection(self) -> None:
    # FIX: reset at the top so this is safe to call twice - Stage 10b
    # (resplitting) needs an accurate re-check against the post-split
    # cluster structure, not a stale accumulation from before the split.
    self.conflicts = []
    self.status = "success"
    sizes = [len(v) for v in self.clusters.values()]
    if len(sizes) >= 2:
      mean_sz = sum(sizes) / len(sizes)
      std_sz = (sum((s - mean_sz) ** 2 for s in sizes) / len(sizes)) ** 0.5
    else:
      mean_sz = sizes[0] if sizes else 1
      std_sz = 1.0
    oversized_threshold = mean_sz + OVERSIZED_CLUSTER_FACTOR * std_sz

    for cid, members in self.clusters.items():
      obs_list = [self._obs_by_id[oid] for oid in members if oid in self._obs_by_id]
      loc_time_map: Dict[Tuple[str, int], Set[str]] = defaultdict(set)
      for obs in obs_list:
        if obs.location.strip():
          loc_time_map[(obs.location.strip().lower(), obs.time_offset_sec)].add(obs.entity_norm)
      for (loc, ts), _ in loc_time_map.items():
        if any(o.time_offset_sec == ts and o.location.strip() and o.location.strip().lower() != loc for o in obs_list):
          self.conflicts.append({
            "type": "physical_impossibility",
            "cluster_id": cid,
            "detail": f"Concurrent different locations at t={ts}s in {cid}.",
          })
          self.status = "awaiting_human_validation"
          break
      cc = self._cluster_confidence(members)
      if cc < CLUSTER_CONFIDENCE_FLOOR:
        self.conflicts.append({
          "type": "low_confidence",
          "cluster_id": cid,
          "cluster_confidence": round(cc, 4),
          "detail": f"Cluster {cid} confidence {cc:.3f} below floor.",
        })
        self.status = "awaiting_human_validation"
      if oversized_threshold > 0 and len(members) > oversized_threshold:
        self.conflicts.append({
          "type": "oversized_cluster",
          "cluster_id": cid,
          "size": len(members),
          "threshold": round(oversized_threshold, 1),
          "detail": f"Cluster {cid} oversized ({len(members)}).",
        })
        self.status = "awaiting_human_validation"

    # FIX: real over-merging on CASE_ATM_002 (verified against ground
    # truth) went undetected end-to-end - confidently reported 2 entities
    # when the FIR itself claims 3 suspects + 2 witnesses. The
    # same-modality rule above catches the most direct violations, but
    # cross-modal bridging between genuinely different same-role actors
    # coordinating closely in time remains a hard, open problem with the
    # current feature set. Rather than pretend a threshold tweak reliably
    # solves that, this makes the discrepancy VISIBLE: compare resolved
    # entity counts per role against the FIR own stated counts
    # (legitimate input, not ground truth) and flag a mismatch, so it
    # surfaces as AMBIGUOUS for investigator review instead of a
    # confident, wrong answer.
    if self.fir_role_counts:
      resolved_role_counts: Dict[str, Set[str]] = defaultdict(set)
      for cid, members in self.clusters.items():
        obs_list = [self._obs_by_id[oid] for oid in members if oid in self._obs_by_id]
        for obs in obs_list:
          resolved_role_counts[obs.role.strip().lower()].add(cid)
      for role, expected_count in self.fir_role_counts.items():
        role_key = str(role).strip().lower()
        actual_count = len(resolved_role_counts.get(role_key, set()))
        if actual_count > 0 and actual_count < int(expected_count):
          self.conflicts.append({
            "type": "role_count_mismatch",
            "role": role_key,
            "expected_count": int(expected_count),
            "resolved_count": actual_count,
            "detail": (
              f"FIR states {expected_count} distinct '{role_key}' role(s), "
              f"but only {actual_count} were resolved. This can mean the "
              "clustering over-merged distinct people, OR that some FIR-claimed "
              "actors simply have no observations in this evidence set (a "
              "legitimate gap, not an error) - the pipeline cannot distinguish "
              "the two without ground truth. Review manually."
            ),
          })
          self.status = "awaiting_human_validation"

  def _stage_10b_constrained_resplit(self) -> None:
    """
    FIX: attempts to algorithmically resolve role_count_mismatch instead of
    only flagging it, using single-linkage clustering (build a maximum
    spanning tree over the offending cluster's observations by composite
    score already computed - zero new scoring, zero new API calls - then
    cut the weakest bridging edges).

    Safety constraint: a cut is only applied if EVERY edge being cut scores
    below SPLIT_CUT_MAX_WEIGHT - a genuinely weak bridge, not just the
    least-strong of several confidently strong bonds. Verified against
    CASE_ATM_003 (role_count_mismatch caused by missing data, not
    over-merging - every internal edge is strong, so this correctly
    declines to force a split) and CASE_ATM_002 (the true cross-suspect
    bridge, e.g. speaker_d<->person_50 at 0.751, scores HIGHER than
    legitimate within-suspect bonds like log_30<->speaker_d at 0.681 - so
    this also correctly declines rather than risk cutting the wrong edge).
    """
    self._resplit_log = []
    if not self.fir_role_counts:
      return

    role_to_clusters: Dict[str, Dict[str, List[str]]] = defaultdict(dict)
    for cid, members in self.clusters.items():
      obs_list = [self._obs_by_id[oid] for oid in members if oid in self._obs_by_id]
      for role in {o.role.strip().lower() for o in obs_list}:
        role_to_clusters[role][cid] = members

    for role, expected_count in self.fir_role_counts.items():
      role_key = str(role).strip().lower()
      clusters_for_role = role_to_clusters.get(role_key, {})
      resolved_count = len(clusters_for_role)
      deficit = int(expected_count) - resolved_count
      if deficit <= 0 or not clusters_for_role:
        continue

      target_cid = max(clusters_for_role, key=lambda c: len(clusters_for_role[c]))
      target_members = clusters_for_role[target_cid]
      target_subclusters = deficit + 1

      new_groups = self._attempt_single_linkage_split(target_members, target_subclusters)
      if new_groups is None:
        continue

      del self.clusters[target_cid]
      for i, group in enumerate(new_groups):
        self.clusters[f"{target_cid}s{i + 1}"] = group
      self._resplit_log.append({
        "original_cluster": target_cid,
        "role": role_key,
        "split_into": len(new_groups),
        "fir_expected_count": int(expected_count),
      })

    if self._resplit_log:
      log.info("Stage 10b: applied %d resplit(s): %s", len(self._resplit_log), self._resplit_log)
      self._stage_10_conflict_detection()

  def _attempt_single_linkage_split(self, obs_ids: List[str], k: int):
    """
    Build a maximum spanning tree over obs_ids (edge weight = best
    composite score already computed for the pair), then cut the (k-1)
    weakest tree edges to yield k components. Returns None if there aren't
    enough observations, the graph isn't connected, or any candidate cut
    edge is not weak enough to justify treating it as a genuine boundary
    between different people.
    """
    if len(obs_ids) < k:
      return None

    obs_set = set(obs_ids)
    G = nx.Graph()
    G.add_nodes_from(obs_ids)
    for pf in self._pair_features.values():
      a, b = pf.obs_a.obs_id, pf.obs_b.obs_id
      if a in obs_set and b in obs_set and a != b:
        w = pf.composite
        if G.has_edge(a, b):
          G[a][b]["weight"] = max(G[a][b]["weight"], w)
        else:
          G.add_edge(a, b, weight=w)

    by_alias: Dict[str, List[str]] = defaultdict(list)
    for oid in obs_ids:
      by_alias[self._obs_by_id[oid].entity_norm].append(oid)
    for alias_group in by_alias.values():
      for i in range(len(alias_group)):
        for j in range(i + 1, len(alias_group)):
          a, b = alias_group[i], alias_group[j]
          if G.has_edge(a, b):
            G[a][b]["weight"] = max(G[a][b]["weight"], 1.0)
          else:
            G.add_edge(a, b, weight=1.0)

    if G.number_of_edges() == 0 or not nx.is_connected(G):
      return None

    mst = nx.maximum_spanning_tree(G, weight="weight")
    mst_edges = sorted(mst.edges(data="weight"), key=lambda e: e[2])
    cut_edges = mst_edges[: k - 1]

    if any(w >= SPLIT_CUT_MAX_WEIGHT for _, _, w in cut_edges):
      return None

    mst.remove_edges_from([(a, b) for a, b, _ in cut_edges])
    components = list(nx.connected_components(mst))
    if len(components) != k:
      return None
    return [sorted(c) for c in components]

  def _stage_11_entity_labeling(self) -> None:
    self._canonical_entities = []
    assigned: Set[str] = set()
    deduped: Dict[str, List[str]] = {}
    for cid, members in sorted(self.clusters.items()):
      unique = [m for m in members if m not in assigned]
      if unique:
        deduped[cid] = unique
        assigned |= set(unique)
    self.clusters = deduped

    def fmt_ts(epoch: float) -> str:
      return "" if epoch <= 0 else datetime.fromtimestamp(epoch, tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    for entity_idx, (_, members) in enumerate(self.clusters.items(), start=1):
      obs_list = [self._obs_by_id[oid] for oid in members if oid in self._obs_by_id]
      if not obs_list:
        continue
      raw_aliases = [o.entity for o in obs_list]
      primary_alias = Counter(raw_aliases).most_common(1)[0][0]
      aliases_unique = list(dict.fromkeys(raw_aliases))
      epochs = [o._ts_epoch for o in obs_list if o._ts_epoch > 0]
      member_set = set(members)
      confirmed_obs_ids: Set[str] = set()
      for e in self.edges:
        if e.classification == "confirmed" and (e.obs_id_1 in member_set or e.obs_id_2 in member_set):
          if e.obs_id_1 in member_set:
            confirmed_obs_ids.add(e.obs_id_1)
          if e.obs_id_2 in member_set:
            confirmed_obs_ids.add(e.obs_id_2)
      candidate_mentions: List[Dict[str, Any]] = []
      seen: Set[Tuple[str, str]] = set()
      for oid in members:
        for cdata in self._candidate_data.get(oid, []):
          key = (cdata["candidate_alias"], str(round(cdata["score"], 2)))
          if key not in seen:
            candidate_mentions.append({
              "candidate_alias": cdata["candidate_alias"],
              "score": cdata["score"],
              "reasons": list(cdata.get("reasons", [])),
            })
            seen.add(key)
      self._canonical_entities.append({
        "entity_id": f"entity_{entity_idx}",
        "aliases": aliases_unique,
        "primary_alias": primary_alias,
        "total_mentions": len(obs_list),
        "confirmed_mentions": sorted(confirmed_obs_ids),
        "candidate_mentions": candidate_mentions,
        "confidence_score": round(self._cluster_confidence(members), 4),
        "confirmed_edges": sum(1 for e in self.edges if e.classification == "confirmed" and (e.obs_id_1 in member_set or e.obs_id_2 in member_set)),
        "candidate_edges": sum(1 for e in self.edges if e.classification in ("likely", "possible") and not e.hard_negative and (e.obs_id_1 in member_set or e.obs_id_2 in member_set)),
        "modalities": sorted({o.modality for o in obs_list}),
        "locations": sorted({o.location for o in obs_list if o.location}),
        "roles": sorted({o.role for o in obs_list}),
        "sources": sorted({o.obs_id for o in obs_list}),
        "earliest_timestamp": fmt_ts(min(epochs) if epochs else 0),
        "latest_timestamp": fmt_ts(max(epochs) if epochs else 0),
        "time_span_seconds": int(max(epochs) - min(epochs)) if epochs else 0,
      })

  def _stage_12_package(self, total_time: float) -> Dict[str, Any]:
    cluster_output = []
    for cid, members in self.clusters.items():
      member_set = set(members)
      edge_dicts = []
      for e in self.edges:
        if e.obs_id_1 in member_set or e.obs_id_2 in member_set:
          edge_reasons = [] if e.hard_negative or e.classification == "rejected" else list(e.reasons)
          edge_dicts.append({
            "alias_1": e.alias_1,
            "alias_2": e.alias_2,
            "weight": round(e.weight, 4),
            "support": e.support,
            "classifications": {
              "confirmed": 1 if e.classification == "confirmed" else 0,
              "candidate": 1 if e.classification in ("likely", "possible") else 0,
              "rejected": 1 if e.classification == "rejected" else 0,
            },
            "hard_negative": e.hard_negative,
            "reasons": edge_reasons,
          })
      cluster_output.append({
        "cluster_id": cid,
        "size": len(members),
        "aliases": list({self._obs_by_id[oid].entity for oid in members if oid in self._obs_by_id}),
        "obs_ids": sorted(members),
        "edges": edge_dicts,
      })

    remap = {
      "stage_1_intake": "stage_1_intake",
      "stage_2_normalization": "stage_2_normalization",
      "stage_3_blocking": "stage_3_blocking",
      "stage_4_feature_computation": "stage_4_features",
      "stage_5_scoring": "stage_5_scoring",
      "stage_6_classification": "stage_6_classification",
      "stage_7_graph_construction": "stage_7_graph_building",
      "stage_8_clustering": "stage_8_clustering",
      "stage_9_guarded_attachment": "stage_9_attachment",
      "stage_10_conflict_detection": "stage_10_conflict_detection",
      "stage_11_entity_labeling": "stage_11_labeling",
    }
    final_timings = {remap.get(k, k): round(v, 6) for k, v in self.stage_timings.items()}
    final_timings["stage_12_packaging"] = 0.0

    _output_classification, _output_classification_reason = classify_resolution_output(
      self._canonical_entities, self.conflicts
    )
    return {
      "case_id": self.case_id,
      "status": self.status,
      "error_message": self.error_message,
      "entity_count": len(self._canonical_entities),
      "canonical_entities": self._canonical_entities,
      "clusters": cluster_output,
      "conflicts_detected": len(self.conflicts),
      # FIX (Timeline Agent contract bug): expose the actual conflict
      # records, not just their count. Without this, the Timeline Agent
      # (or any downstream consumer) cannot know WHICH observations were
      # flagged, and conflict-aware confidence penalties silently never fire.
      "conflicts": self.conflicts,
      "output_classification": _output_classification,
      "output_classification_reason": _output_classification_reason,
      # Transparency, mirroring the Timeline Agent notebook's llm_calls_made field.
      "llm_calls_made": self._context_budget.calls_made + self._entity_budget.calls_made,
      "llm_calls_budget": self._context_budget.max_calls + self._entity_budget.max_calls,
      # Transparency: which similarity backend actually scored locations -
      # "embedding" (real semantic model) or "lexical_fallback" (token
      # overlap, used if the model could not be loaded).
      "location_similarity_backend": get_semantic_scorer().backend_used(),
      "resplit_log": self._resplit_log,
      "configuration": {
        "check_duplicates": self.check_duplicates,
        "case_base_time": self.case_base_time,
        "temporal_window_sec": self.temporal_window_sec,
        "max_temporal_gap_sec": self.max_temporal_gap_sec,
        "max_pairs": self.max_pairs,
        "confirmed_threshold": self.confirmed_threshold,
        "candidate_threshold_low": self.candidate_threshold_low,
        "candidate_threshold_high": self.candidate_threshold_high,
        "llm_enabled": self.llm_enabled,
        "groq_model": GROQ_MODEL,
        "cross_modal_merge_min": CROSS_MODAL_MERGE_MIN,
        "merge_composite_min": MERGE_COMPOSITE_MIN,
      },
      "total_processing_time_sec": round(total_time, 6),
      "stage_timings": final_timings,
      "created_at": datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    }


## 10. Public API — `resolve_entities()`


In [ ]:
def resolve_entities(
  observations_payload: Dict[str, Any],
  config: Optional[Dict[str, Any]] = None,
  human_constraints: Optional[HumanConstraints] = None,
  llm_enabled: bool = True,
  max_llm_calls: int = MAX_LLM_CALLS_PER_RUN,
) -> Dict[str, Any]:
  return EntityResolutionPipeline(config=config, human_constraints=human_constraints, llm_enabled=llm_enabled, max_llm_calls=max_llm_calls).run(observations_payload)


## 11a. Built-in test observation payloads


In [ ]:
ATM_DEMO_INPUT = {
  "case_id": "CASE_ATM_004",
  "observations": [
    {"obs_id": "O1", "entity": "Person_97", "role": "suspect", "modality": "video", "location": "ATM booth entrance, exterior facing", "content": "Individual seen walking towards ATM kiosk (captured on footage).", "timestamp": "2024-01-15T10:01:20", "confidence": 0.906},
    {"obs_id": "O2", "entity": "Speaker_K", "role": "suspect", "modality": "audio", "location": "near ATM location, mobile network", "content": "I can see the booth from here.", "timestamp": "2024-01-15T10:01:12", "confidence": 0.613},
    {"obs_id": "O3", "entity": "sms_35", "role": "suspect", "modality": "text", "location": "remote — email server log", "content": "Heading there. Any activity?", "timestamp": "2024-01-15T10:00:55", "confidence": 0.683},
    {"obs_id": "O7", "entity": "Person_97", "role": "suspect", "modality": "video", "location": "ATM booth interior, card reader and keypad area", "content": "Suspect enters the ATM booth unaccompanied (captured on footage).", "timestamp": "2024-01-15T10:03:54", "confidence": 0.875},
    {"obs_id": "O8", "entity": "Speaker_K", "role": "suspect", "modality": "audio", "location": "ATM vicinity, intercepted channel", "content": "I'm in, it's clear.", "timestamp": "2024-01-15T10:03:56", "confidence": 0.778},
    {"obs_id": "O9", "entity": "sms_35", "role": "suspect", "modality": "text", "location": "local police station, complaint desk", "content": "Entered. Starting now.", "timestamp": "2024-01-15T10:04:06", "confidence": 0.730},
    {"obs_id": "O14", "entity": "Speaker_K", "role": "suspect", "modality": "audio", "location": "ATM vicinity, intercepted channel", "content": "Done, I'm coming out.", "timestamp": "2024-01-15T10:04:58", "confidence": 0.836},
    {"obs_id": "O15", "entity": "sms_35", "role": "suspect", "modality": "text", "location": "bank security office, incident registry", "content": "Exited. Don't wait for me.", "timestamp": "2024-01-15T10:04:57", "confidence": 0.785},
    {"obs_id": "O16", "entity": "Person_32", "role": "witness", "modality": "video", "location": "ATM lobby doorway, entry/exit point", "content": "Eyewitness notices a person leaving the ATM enclosure in a rush.", "timestamp": "2024-01-15T10:05:09", "confidence": 0.482},
    {"obs_id": "O17", "entity": "Person_32", "role": "witness", "modality": "video", "location": "ATM booth entrance, exterior facing", "content": "Bystander seen speaking with bank security personnel outside.", "timestamp": "2024-01-15T10:08:44", "confidence": 0.501},
  ],
}

IP_LOG_DEMO = {
  "case_id": "CASE_IP_CROSSMODAL_001",
  "observations": [
    {"obs_id": "T1", "entity": "Suspect A", "role": "suspect", "modality": "text", "location": "interview room transcript", "content": "Suspect A stated they accessed the server at 14:02.", "timestamp": "2024-03-01T14:05:00", "confidence": 0.82},
    {"obs_id": "L1", "entity": "192.168.4.22", "role": "suspect", "modality": "text", "location": "firewall log / dmz segment", "content": "SSH login accepted from 192.168.4.22 at 14:02:11.", "timestamp": "2024-03-01T14:02:11", "confidence": 0.91},
    {"obs_id": "V1", "entity": "individual_in_red_jacket", "role": "suspect", "modality": "video", "location": "server room hallway camera 3", "content": "Individual in red jacket badged into server room at 14:01:50.", "timestamp": "2024-03-01T14:01:50", "confidence": 0.88},
    {"obs_id": "W1", "entity": "Security Guard", "role": "witness", "modality": "audio", "location": "server room entrance", "content": "Guard reports unknown person in red jacket near rack 4.", "timestamp": "2024-03-01T14:03:00", "confidence": 0.74},
  ],
}


## 11a2. (Optional) Upload your own case file

Run this cell to upload your own `*_obs_only.json` observation file. **Skip this cell** to use the built-in demo cases instead - the next cell falls back to them automatically if nothing was uploaded.

In [ ]:
UPLOADED_CASE = None

try:
    from google.colab import files  # type: ignore
    print("Select your obs_only.json file...")
    uploaded = files.upload()
    for name, content in uploaded.items():
        data = json.loads(content.decode("utf-8"))
        if "observations" in data:
            UPLOADED_CASE = data
            print(f"Loaded '{name}' (case_id={data.get('case_id', '?')}, "
                  f"{len(data['observations'])} observations)")
        else:
            print(f"'{name}' has no 'observations' key - ignoring.")
except ImportError:
    print("Not running in Colab - skipping upload prompt.\n"
          "If running locally, load your file manually instead, e.g.:\n"
          "    UPLOADED_CASE = json.load(open('my_case_obs_only.json'))")


Select your obs_only.json file...


Saving CASE_ATM_002_obs_only.json to CASE_ATM_002_obs_only.json
Loaded 'CASE_ATM_002_obs_only.json' (case_id=CASE_ATM_002, 17 observations)


## 11b. Demo & test runner

Runs built-in **ATM** and **IP/cross-modal** scenarios. Optionally set `CASE_FILE_PATH` to a `*_obs_only.json` file.


In [ ]:
import os, sys, json

# Optional: path to your observation JSON file (local/non-Colab use)
CASE_FILE_PATH = os.environ.get('FORENSYNTH_CASE_FILE')  # e.g. r'C:\\data\\CASE_ATM_004_obs_only.json'

cases = [('ATM multi-modal suspect', ATM_DEMO_INPUT), ('IP + transcript + video', IP_LOG_DEMO)]
if UPLOADED_CASE is not None:
    cases = [('uploaded case', UPLOADED_CASE)]
elif CASE_FILE_PATH and os.path.exists(CASE_FILE_PATH):
    with open(CASE_FILE_PATH, encoding='utf-8') as fh:
        cases = [('custom file', json.load(fh))]

LLM_ON = bool(os.environ.get('GROQ_API_KEY'))
print(f'LLM enabled: {LLM_ON}')

for label, payload in cases:
    print('\n' + '=' * 72)
    print(f'CASE: {label} ({payload["case_id"]})')
    result = resolve_entities(payload, llm_enabled=LLM_ON)
    print(f'status={result["status"]} entities={result["entity_count"]} conflicts={result["conflicts_detected"]}')
    print(f'output_classification={result["output_classification"]}  ({result["output_classification_reason"]})')
    print(f'llm_calls_made={result["llm_calls_made"]}/{result["llm_calls_budget"]}')
    print(f'location_similarity_backend={result["location_similarity_backend"]}')
    for ent in result['canonical_entities']:
        print(f"  - {ent['entity_id']}: aliases={ent['aliases']} modalities={ent['modalities']}")
    if payload['case_id'] == 'CASE_ATM_004':
        assert result['entity_count'] <= 3
        suspect = next(e for e in result['canonical_entities'] if 'suspect' in e['roles'])
        assert len(suspect['aliases']) >= 2
    if payload['case_id'] == 'CASE_IP_CROSSMODAL_001':
        suspects = [e for e in result['canonical_entities'] if 'suspect' in e['roles']]
        assert len(suspects) == 1 and len(suspects[0]['aliases']) >= 2
print('\n[OK] Demo assertions passed.')



LLM enabled: True

CASE: uploaded case (CASE_ATM_002)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

status=awaiting_human_validation entities=2 conflicts=3
output_classification=AMBIGUOUS  (avg_confidence=0.75, conflict_fraction=1.50, low_confidence_fraction=0.00 - at least one signal crossed the ambiguity threshold; treat these entity mappings as low-trust and prioritise investigator review before downstream use.)
llm_calls_made=2/2
location_similarity_backend=embedding
  - entity_1: aliases=['Speaker_Q', 'Speaker_D', 'log_30', 'Person_50', 'Person_97'] modalities=['audio', 'text', 'video']
  - entity_2: aliases=['Speaker_K', 'email_68', 'sms_74'] modalities=['audio', 'text']

[OK] Demo assertions passed.


## 12. Timeline Agent payload

Extracts the contract fields your **Timeline Agent** consumes.


In [ ]:
timeline_agent_payload = {
    'case_id': result['case_id'],
    'canonical_entities': result['canonical_entities'],
    'clusters': result['clusters'],
    'conflicts_detected': result['conflicts_detected'],
    # FIX: pass the actual conflict records through, not just the count -
    # this is what lets the Timeline Agent localise conflicts to specific
    # observations/events instead of only knowing a count occurred.
    'conflicts': result['conflicts'],
}
print(json.dumps(timeline_agent_payload, indent=2))



{
  "case_id": "CASE_ATM_002",
  "canonical_entities": [
    {
      "entity_id": "entity_1",
      "aliases": [
        "Speaker_Q",
        "Speaker_D",
        "log_30",
        "Person_50",
        "Person_97"
      ],
      "primary_alias": "Speaker_D",
      "total_mentions": 14,
      "confirmed_mentions": [
        "O10",
        "O5",
        "O9"
      ],
      "candidate_mentions": [],
      "confidence_score": 0.8133,
      "confirmed_edges": 2,
      "candidate_edges": 74,
      "modalities": [
        "audio",
        "text",
        "video"
      ],
      "locations": [
        "ATM booth entrance, exterior facing",
        "ATM booth interior, card reader and keypad area",
        "ATM kiosk exterior, street-side view",
        "ATM street frontage, wide-angle coverage",
        "ATM vicinity, bystander position",
        "ATM vicinity, intercepted channel",
        "near ATM location, mobile network",
        "remote \u2014 email server log"
      ],
      "roles": [
 

## 12b. Save this as a file to feed into the Timeline Agent notebook

The Timeline Agent notebook's upload cell ("Now select the matching entity-resolution output JSON for the same case...") needs an actual `.json` **file**, not the text printed above. This cell writes `timeline_agent_payload` to a real file, named after the case, and downloads it automatically if you're in Colab - upload that downloaded file when the Timeline Agent notebook asks for it.

In [ ]:
_er_output_filename = f"{timeline_agent_payload['case_id']}_er_output.json"
with open(_er_output_filename, "w") as f:
    json.dump(timeline_agent_payload, f, indent=2)
print(f"Saved {_er_output_filename}")

try:
    from google.colab import files  # type: ignore
    files.download(_er_output_filename)
    print("Download started - this is the file to upload in the Timeline Agent notebook.")
except ImportError:
    print(f"Not running in Colab - find '{_er_output_filename}' in the current working directory instead.")


Saved CASE_ATM_002_er_output.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started - this is the file to upload in the Timeline Agent notebook.


## 13. Validate output contract


In [ ]:
EXPECTED_TIMING_KEYS = {
    'stage_1_intake', 'stage_2_normalization', 'stage_3_blocking', 'stage_4_features',
    'stage_5_scoring', 'stage_6_classification', 'stage_7_graph_building', 'stage_8_clustering',
    'stage_9_attachment', 'stage_10_conflict_detection', 'stage_11_labeling', 'stage_12_packaging',
}
actual = set(result['stage_timings'].keys())
assert EXPECTED_TIMING_KEYS <= actual, f'missing keys: {EXPECTED_TIMING_KEYS - actual}'
print('[OK] All stage_timings keys present.')



[OK] All stage_timings keys present.
